# Real Traffic Variable-Lag Target Relevance

Real-world variable-lag feature selection experiment using **METR-LA** traffic sensor data.

Reuses the full synthetic pipeline infrastructure:
LeakageChecks · ClassicalBaselines · ChronosEmbeddingExtractor · MultiPoolingExtractor ·
RepresentationMetrics (+ centered cosine variants) · NegativeControlAdjustment ·
Evaluator · Reporter · Visualizer · ExperimentRunner

**Dataset:** `METR-LA.zip` in the current directory (207 sensors, 5-min intervals, ~4 months)  
**Labels:** `proxy_relevant` (graph-informed, **not** verified causal), `structured_irrelevant`, `background_irrelevant`

> Required kernel: `yael_env` (chronos-2, torch+CUDA, h5py)


In [ ]:
import os, time, datetime, dataclasses, pathlib, hashlib, pickle, warnings
import json as _json
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
print('Imports OK')


## 1. Configuration

In [ ]:
@dataclasses.dataclass
class TrafficConfig:
    NOTEBOOK_NAME:   str = 'Real_Traffic_Variable_Lag_Target_Relevance.ipynb'
    DATASET_NAME:    str = 'METR_LA'
    ZIP_PATH:        str = 'METR-LA.zip'          # path to METR-LA.zip
    H5_KEY:          str = 'METR-LA.h5'           # filename inside zip
    ADJ_KEY:         str = 'adj_METR-LA.pkl'      # filename inside zip

    # Context lengths (equiv. to T_VALUES in synthetic)
    CONTEXT_LENGTHS: Tuple[int, ...] = (128, 256)

    # max_lag = int(LAG_MAX_FRAC * context_length)  — same as synthetic
    LAG_MAX_FRAC: float = 0.15

    # 12 features per instance
    N_PROXY_RELEVANT: int = 3
    N_STRUCTURED_IRR: int = 3
    N_BACKGROUND_IRR: int = 6
    N_FEATURES:       int = 12

    # Run scale
    TARGET_SENSOR_COUNT:    int          = 50
    WINDOWS_PER_TARGET:     int          = 3
    STRIDE_BETWEEN_WINDOWS: Optional[int]= None   # None → context_length // 2

    # Missing values (METR-LA has none, kept for robustness)
    MAX_MISSINGNESS_FRAC: float = 0.10

    # Chronos
    MODEL_ID: str = 'amazon/chronos-2'

    # Full layer + pooling suite
    SINGLE_LAYERS:      Tuple[int,...]           = (1,3,6,9,11)
    LAYER_RANGES:       Tuple[Tuple[int,int],...]= ((3,6),(6,9),(3,10))
    POOLING_STRATEGIES: Tuple[str,...]           = ('mean_pooling','max_pooling','std_pooling','norm_weighted_pooling')

    NORMALIZE_FEATURES:  bool  = True
    OUTPUT_PREFIX:       str   = 'Real_Traffic_Variable_Lag_Analysis'
    RUNTIME_THRESHOLD_S: float = 6*3600.0


CFG = TrafficConfig()
print(f'Notebook       : {CFG.NOTEBOOK_NAME}')
print(f'Dataset        : {CFG.DATASET_NAME}  ({CFG.ZIP_PATH})')
print(f'Context lengths: {CFG.CONTEXT_LENGTHS}')
print(f'Target sensors : {CFG.TARGET_SENSOR_COUNT}')
print(f'Windows/target : {CFG.WINDOWS_PER_TARGET}')
print(f'Features/inst  : {CFG.N_FEATURES}  (proxy={CFG.N_PROXY_RELEVANT}, str={CFG.N_STRUCTURED_IRR}, bg={CFG.N_BACKGROUND_IRR})')
print(f'Layers         : single={CFG.SINGLE_LAYERS}  ranges={CFG.LAYER_RANGES}')
print(f'Poolings       : {CFG.POOLING_STRATEGIES}')


## 2. METR-LA Dataset Loader

Loads directly from `METR-LA.zip` in the current directory.
No extraction needed — reads `METR-LA.h5` and `adj_METR-LA.pkl` from the zip.

| File in zip | Description |
|-------------|-------------|
| `METR-LA.h5` | Speed matrix `(34272 timesteps × 207 sensors)`, 5-min intervals, 2012 |
| `adj_METR-LA.pkl` | Weighted adjacency `(207×207)`, Gaussian kernel of road distance |


In [ ]:
class MetrLaLoader:
    """
    Loads METR-LA directly from the zip archive (no extraction required).
    Uses h5py to parse the HDF5 file (pytables/tables not required).

    After load():
      .speed_df   : pd.DataFrame  shape (34272, 207), datetime index, sensor-ID columns
      .adj        : np.ndarray    shape (207, 207), Gaussian-weighted adjacency
      .sensor_ids : list of 207 sensor ID strings
      .sensor_map : dict  sensor_id -> column index
    """

    def __init__(self, zip_path='METR-LA.zip', h5_key='METR-LA.h5', adj_key='adj_METR-LA.pkl'):
        self.zip_path  = pathlib.Path(zip_path)
        self.h5_key    = h5_key
        self.adj_key   = adj_key
        self.speed_df   = None
        self.adj        = None
        self.sensor_ids = None
        self.sensor_map = None
        self._loaded    = False

    def load(self):
        import zipfile, tempfile, h5py
        print(f'Loading METR-LA from: {self.zip_path}')
        if not self.zip_path.exists():
            raise FileNotFoundError(f'Zip not found: {self.zip_path}')

        with zipfile.ZipFile(self.zip_path) as z:
            # ── Load speed matrix ─────────────────────────────────────────
            with z.open(self.h5_key) as zf:
                raw = zf.read()
            with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as tmp:
                tmp.write(raw); tmppath = tmp.name
            try:
                with h5py.File(tmppath, 'r') as f:
                    sensor_bytes = f['df/axis0'][:]          # (207,) bytes
                    timestamps   = f['df/axis1'][:]          # (34272,) int64 ns
                    values       = f['df/block0_values'][:]  # (34272, 207) float64
            finally:
                os.unlink(tmppath)

            self.sensor_ids = [s.decode() for s in sensor_bytes]
            self.sensor_map = {sid: i for i, sid in enumerate(self.sensor_ids)}
            ts_index        = pd.DatetimeIndex(timestamps)   # ns epoch → datetime
            self.speed_df   = pd.DataFrame(
                values, index=ts_index, columns=self.sensor_ids
            )

            # ── Load adjacency matrix ─────────────────────────────────────
            with z.open(self.adj_key) as zf:
                obj = pickle.load(zf, encoding='latin1')
            # obj is a list: [sensor_id_list, sensor_map_dict, adj_matrix]
            self.adj = np.array(obj[2], dtype=float)  # (207, 207)

        self._loaded = True
        self._report_stats()
        return self

    @property
    def loaded(self): return self._loaded

    def _report_stats(self):
        df = self.speed_df
        print(f'  Shape     : {df.shape}  (timesteps × sensors)')
        print(f'  Sensors   : {df.shape[1]}')
        print(f'  Timesteps : {df.shape[0]}')
        if len(df) > 1:
            print(f'  Interval  : {pd.Series(df.index).diff().median()}')
            print(f'  Range     : {df.index[0]}  →  {df.index[-1]}')
        miss = df.isna().mean()
        print(f'  Missing   : {miss.mean()*100:.4f}% mean  (max: {miss.max()*100:.4f}%)')
        print(f'  Speed     : [{df.values.min():.1f}, {df.values.max():.1f}] mph')
        print(f'  Adj matrix: {self.adj.shape}  nonzero={( self.adj>0).mean():.4f}')
        print(f'  Sensor IDs (first 5): {self.sensor_ids[:5]}')


_loader = MetrLaLoader(CFG.ZIP_PATH, CFG.H5_KEY, CFG.ADJ_KEY)
_loader.load()

SPEED_DF    = _loader.speed_df
ADJ_MATRIX  = _loader.adj
SENSOR_IDS  = _loader.sensor_ids
SENSOR_MAP  = _loader.sensor_map
print(f'Dataset ready: {SPEED_DF.shape[1]} sensors × {SPEED_DF.shape[0]} timesteps')


## 3. Sensor Selection

Uses the **weighted adjacency matrix** (Gaussian kernel of road distance):
- `proxy_relevant` (3): highest-weight neighbours (adj > 0, sorted descending)
- `structured_irrelevant` (3): moderate-weight neighbours (lower end of connected)
- `background_irrelevant` (6): sensors with zero adjacency weight (distant)

Labels are **proxy relevance** — not verified causal ground truth.


In [ ]:
class SensorSelector:
    METADATA_NOTE = (
        'Real-world labels are graph-informed proxy relevance labels, '
        'not verified causal ground truth.'
    )

    def __init__(self, n_sensors, cfg, adj=None, speed_values=None):
        self.n     = n_sensors
        self.cfg   = cfg
        self.adj   = adj          # (n, n) weighted adjacency; None → use correlation
        self._vals = speed_values # (T, n) for correlation fallback
        self._corr = None

    def _get_corr(self):
        if self._corr is None:
            arr = self._vals.copy()
            meds = np.nanmedian(arr, axis=0)
            for j in range(arr.shape[1]):
                arr[np.isnan(arr[:,j]),j] = meds[j]
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self._corr = np.nan_to_num(np.corrcoef(arr.T), nan=0.0)
        return self._corr

    def select_candidates(self, tgt_idx):
        np_ = self.cfg.N_PROXY_RELEVANT
        ns  = self.cfg.N_STRUCTURED_IRR
        nb  = self.cfg.N_BACKGROUND_IRR

        if self.adj is not None:
            row = self.adj[tgt_idx].copy().astype(float)
            row[tgt_idx] = -1.0   # exclude self
            order = np.argsort(-row)  # descending weight
            pos_mask = row[order] > 0
            pos_idxs = order[pos_mask].tolist()
            neg_idxs = order[~pos_mask].tolist()
            # proxy: top-np_ by weight (positive)
            proxy  = pos_idxs[:np_]
            # structured: lower end of positive neighbours (moderate weight)
            struct_start = max(np_, len(pos_idxs) - ns - nb)
            struct = pos_idxs[struct_start:struct_start+ns]
            # background: sensors with zero adjacency (truly distant)
            bg = neg_idxs[:nb]
        else:
            corr = np.abs(self._get_corr()[tgt_idx]).copy()
            corr[tgt_idx] = -1.0
            order = np.argsort(-corr)
            n_avail = self.n - 1
            proxy  = list(order[:np_])
            mid    = max(np_, n_avail//4)
            struct = list(order[mid:mid+ns])
            bg     = list(order[max(mid+ns, n_avail-nb):n_avail-nb+nb])

        # Deduplicate and pad
        proxy, struct, bg = self._dedup(list(range(self.n)), proxy, struct, bg, np_, ns, nb, tgt_idx)
        return {'proxy_relevant':proxy,'structured_irrelevant':struct,'background_irrelevant':bg}

    @staticmethod
    def _dedup(all_idxs, proxy, struct, bg, np_, ns, nb, exclude):
        def fill(lst, n, used):
            res = [x for x in lst if x not in used]
            for i in all_idxs:
                if len(res)>=n: break
                if i not in used and i not in res: res.append(i)
            return res[:n]
        used = {exclude}
        proxy  = fill(proxy,  np_, used); used.update(proxy)
        struct = fill(struct, ns,  used); used.update(struct)
        bg     = fill(bg,     nb,  used)
        return proxy, struct, bg


print('SensorSelector ready.')
print(f'  Uses adj matrix (Gaussian-weighted road distances).')
print(f'  proxy_relevant: top-3 by adjacency weight')
print(f'  structured_irrelevant: lower connected neighbours')
print(f'  background_irrelevant: zero-adjacency sensors')


## 4. FeatureMetadata and InstanceData

Same dataclasses as synthetic notebook. `is_causal = (feature_role == 'proxy_relevant')`
for Evaluator compatibility. Plots use *Proxy Recovery@3* / *Mean Relevant Rank*.


In [ ]:
@dataclasses.dataclass
class FeatureMetadata:
    feature_name:          str
    concept_type:          str    # sensor ID
    feature_role:          str    # proxy_relevant | structured_irrelevant | background_irrelevant
    causal_weight:         float  # 1.0 for proxy_relevant, 0.0 otherwise
    is_matched_distractor: bool
    note:                  str = ''


@dataclasses.dataclass
class InstanceData:
    T:                int                    # = context_length + max_lag
    seed:             int                    # target column index
    noise_level:      float                  # missingness rate (0.0 for METR-LA)
    causal_setting:   str                    # 'real_traffic_proxy_labels'
    instance_id:      int                    # window index
    features:         Dict[str, np.ndarray]  # feature_name -> (T,)
    Y_clean:          np.ndarray             # (T,)
    Y:                np.ndarray             # (T,) NaN before valid_start
    metadata:         List[FeatureMetadata]
    max_lag:          int
    lag_arrays:       Dict[str, np.ndarray]  # {} for real data
    valid_start:      int                    # = max_lag
    window_start_idx: int  = 0
    target_sensor:    str  = ''
    context_length:   int  = 0
    missingness_rate: float = 0.0


print('FeatureMetadata and InstanceData ready.')
print('  is_causal = (feature_role in (causal, proxy_relevant)) for Evaluator')


## 5. TrafficVariableLagDatasetBuilder

In [ ]:
class TrafficVariableLagDatasetBuilder:
    """
    Converts the METR-LA DataFrame into InstanceData objects compatible with the
    existing pipeline.

    Segment: window_start → window_start + context_length + max_lag
    valid_start = max_lag  (consistent with synthetic convention)
    Y[:max_lag] = NaN      (allows lagged alignment without padding)
    lag_arrays = {}        (true lags unknown for real data)
    """

    def __init__(self, speed_df, cfg, adj=None):
        self.speed_df = speed_df
        self.cfg      = cfg
        self._columns = list(speed_df.columns)
        self._values  = speed_df.values.astype(float)  # (T_total, n_sensors)
        self._n_time  = len(speed_df)
        self.selector = SensorSelector(
            n_sensors=speed_df.shape[1], cfg=cfg,
            adj=adj, speed_values=self._values
        )
        self._cand_cache: Dict[int, dict] = {}

    def build_instance(self, target_col_idx, context_length, window_start):
        """Returns InstanceData or None if segment has too many missing values."""
        max_lag = int(self.cfg.LAG_MAX_FRAC * context_length)
        seg_len = context_length + max_lag
        seg_end = window_start + seg_len
        if seg_end > self._n_time: return None

        target_col = self._columns[target_col_idx]
        if target_col_idx not in self._cand_cache:
            self._cand_cache[target_col_idx] = self.selector.select_candidates(target_col_idx)
        candidates = self._cand_cache[target_col_idx]

        Y_raw = self._values[window_start:seg_end, target_col_idx].copy()
        if np.isnan(Y_raw).mean() > self.cfg.MAX_MISSINGNESS_FRAC: return None

        features: Dict[str, np.ndarray] = {}
        metadata: List[FeatureMetadata] = []
        miss_rates = [float(np.isnan(Y_raw).mean())]

        role_map = [
            ('proxy_relevant',        candidates['proxy_relevant'],        1.0),
            ('structured_irrelevant', candidates['structured_irrelevant'], 0.0),
            ('background_irrelevant', candidates['background_irrelevant'], 0.0),
        ]
        feat_idx = 0
        for role, col_idxs, cw in role_map:
            for ci in col_idxs:
                raw = self._values[window_start:seg_end, ci].copy()
                miss = float(np.isnan(raw).mean())
                if miss > self.cfg.MAX_MISSINGNESS_FRAC: return None
                miss_rates.append(miss)
                raw = self._ffill(raw)
                if self.cfg.NORMALIZE_FEATURES: raw = self._zscore(raw)
                fn  = f'f{feat_idx:02d}_{role[:3]}_{self._columns[ci]}'
                features[fn] = raw
                metadata.append(FeatureMetadata(
                    feature_name=fn, concept_type=str(self._columns[ci]),
                    feature_role=role, causal_weight=cw,
                    is_matched_distractor=(role=='structured_irrelevant'),
                    note=SensorSelector.METADATA_NOTE,
                ))
                feat_idx += 1

        Y_fill = self._ffill(Y_raw)
        if self.cfg.NORMALIZE_FEATURES: Y_fill = self._zscore(Y_fill)
        Y_out = Y_fill.copy()
        Y_out[:max_lag] = np.nan
        for fn in features: features[fn][:max_lag] = np.nan

        return InstanceData(
            T=seg_len, seed=target_col_idx, noise_level=float(np.mean(miss_rates)),
            causal_setting='real_traffic_proxy_labels', instance_id=window_start,
            features=features, Y_clean=Y_fill.copy(), Y=Y_out,
            metadata=metadata, max_lag=max_lag, lag_arrays={},
            valid_start=max_lag,
            window_start_idx=window_start, target_sensor=str(target_col),
            context_length=context_length, missingness_rate=float(np.mean(miss_rates)),
        )

    def iter_instances(self, target_col_idxs, context_length, windows_per_target, stride=None):
        max_lag = int(self.cfg.LAG_MAX_FRAC * context_length)
        seg_len = context_length + max_lag
        if stride is None: stride = context_length // 2
        max_possible = max(1, (self._n_time - seg_len) // stride + 1)
        wpt = min(windows_per_target, max_possible)
        for tci in target_col_idxs:
            for wi in range(wpt):
                inst = self.build_instance(tci, context_length, wi * stride)
                yield tci, wi, inst

    @staticmethod
    def _ffill(x):
        out = x.copy()
        for i in range(1, len(out)):
            if np.isnan(out[i]): out[i] = out[i-1]
        fv = next((v for v in out if not np.isnan(v)), None)
        if fv is not None:
            for i in range(len(out)):
                if np.isnan(out[i]): out[i] = fv
                else: break
        return out

    @staticmethod
    def _zscore(x):
        v = x[~np.isnan(x)]
        if len(v) < 2: return x
        return (x - v.mean()) / (v.std() + 1e-8)


print('TrafficVariableLagDatasetBuilder ready.')
print('  build_instance(target_col_idx, context_length, window_start)')
print('  iter_instances(target_col_idxs, context_length, windows_per_target, stride)')


## 6. LeakageChecks (Real Data)

In [ ]:
class RealDataLeakageChecks:
    """
    Adapted leakage checks for real traffic InstanceData.
    Skips synthetic-only checks (lag_arrays, Y construction).
    """

    @staticmethod
    def run_all(data):
        vs, ml, T = data.valid_start, data.max_lag, data.T
        assert vs == ml, f'[FAIL] valid_start={vs} != max_lag={ml}'
        print(f'  [OK] valid_start={vs} == max_lag={ml}')
        if vs > 0:
            assert np.all(np.isnan(data.Y[:vs])), '[FAIL] Y not NaN before valid_start'
        print(f'  [OK] Y[0:{vs}] = NaN')
        n_nan_y = np.isnan(data.Y[vs:]).sum()
        assert n_nan_y == 0, f'[FAIL] Y has {n_nan_y} NaN after valid_start'
        print(f'  [OK] Y[{vs}:] no NaN ({T-vs} timesteps)')
        for fn, arr in data.features.items():
            n_nan = np.isnan(arr[vs:]).sum()
            assert n_nan == 0, f'[FAIL] Feature {fn} has {n_nan} NaN after valid_start'
        print(f'  [OK] All {len(data.features)} features no NaN after valid_start')
        usable = T - vs
        ws  = usable // 4; wst = max(1, usable//32)
        nw  = max(0, (usable-ws)//wst+1)
        assert nw >= 5, f'[FAIL] n_windows={nw} < 5 (usable_len={usable})'
        print(f'  [OK] Window feasibility: usable={usable}, n_windows≈{nw}')
        print('[RealDataLeakageChecks] All checks passed.')


print('RealDataLeakageChecks ready.')


## 7. Variable-Lag Proxy Diagnostics

Estimates variable-lag structure before running representation metrics.
Computes windowed best-lag and lag variability per (feature, target) pair.
**Diagnostic only.** Does not use future values.


In [ ]:
class LagVariabilityDiagnostics:
    """Cross-correlation and windowed best-lag diagnostics."""

    def __init__(self, max_lag, window_size=None, window_stride=None):
        self.max_lag = max_lag
        self.ws      = window_size
        self.wst     = window_stride

    def compute_for_instance(self, data):
        vs, ml = data.valid_start, data.max_lag
        Y  = data.Y[vs:]; T_v = len(Y)
        ws = self.ws  or max(32, T_v//4)
        wst= self.wst or max(1, T_v//16)
        meta_map = {m.feature_name: m for m in data.metadata}
        rows = []
        for fn, X_full in data.features.items():
            X = X_full[vs:]
            # Global best lag
            g = [self._pearson(X[:T_v-lag] if lag else X, Y[lag:] if lag else Y)
                 for lag in range(ml+1)]
            blg = int(np.argmax(g))
            # Windowed
            wbl = []
            for wstart in range(0, T_v-ws+1, wst):
                Y_w = Y[wstart:wstart+ws]
                wsc = []
                for lag in range(ml+1):
                    x0 = wstart-lag
                    if x0<0: wsc.append(0.); continue
                    X_w = X[x0:x0+ws]
                    wsc.append(self._pearson(X_w, Y_w) if len(X_w)==ws else 0.)
                wbl.append(int(np.argmax(wsc)))
            wbl_arr = np.array(wbl) if wbl else np.array([blg])
            role = meta_map[fn].feature_role if fn in meta_map else 'unknown'
            rows.append({
                'target_sensor':    data.target_sensor,
                'feature_name':     fn,
                'feature_role':     role,
                'context_length':   data.context_length,
                'window_start_idx': data.window_start_idx,
                'best_lag_global':  blg,
                'best_score_global':float(np.max(g)),
                'best_lag_window_mean':  float(wbl_arr.mean()),
                'best_lag_window_std':   float(wbl_arr.std()),
                'best_lag_window_unique_count': int(len(np.unique(wbl_arr))),
                'best_lag_window_range': int(wbl_arr.max()-wbl_arr.min()),
                'lag_variability_score': float(wbl_arr.std()),
                'n_windows_used':   len(wbl_arr),
            })
        return pd.DataFrame(rows)

    @staticmethod
    def _pearson(x, y):
        if len(x)<3 or len(x)!=len(y) or np.std(x)<1e-8 or np.std(y)<1e-8: return 0.
        return float(np.nan_to_num(abs(np.corrcoef(x,y)[0,1]),nan=0.))


print('LagVariabilityDiagnostics ready.')


## 8. Proxy Label Quality Checks

Checks whether proxy_relevant features are distinguishable from irrelevant ones
using lagged Pearson correlation. Warns if labels are weak; does not stop the run.


In [ ]:
class ProxyLabelQualityChecker:
    """Checks separability of proxy labels. Warns but never stops the run."""

    def __init__(self, max_lag): self.max_lag = max_lag

    def check_instance(self, data, verbose=False):
        vs, ml = data.valid_start, data.max_lag
        Y = data.Y[vs:]
        scores_by_role = {'proxy_relevant':[],'structured_irrelevant':[],'background_irrelevant':[]}
        for m in data.metadata:
            X = data.features[m.feature_name][vs:]
            best = max(
                self._pearson(X[:len(X)-lag] if lag else X, Y[lag:] if lag else Y)
                for lag in range(ml+1)
            )
            if m.feature_role in scores_by_role: scores_by_role[m.feature_role].append(best)
        means = {r:float(np.mean(v)) if v else float('nan') for r,v in scores_by_role.items()}
        pm = means.get('proxy_relevant',float('nan'))
        bm = means.get('background_irrelevant',float('nan'))
        sep = not np.isnan(pm) and not np.isnan(bm) and pm > bm
        if verbose:
            if sep: print(f'  [OK]   Proxy labels separable: proxy={pm:.3f} > bg={bm:.3f}')
            else:   print(f'  [WARN] Proxy labels may be weak: proxy={pm:.3f} bg={bm:.3f}')
        return {
            'target_sensor':data.target_sensor,'window_start_idx':data.window_start_idx,
            'context_length':data.context_length,
            'proxy_mean_lagged_pearson':pm,
            'structured_mean_lagged_pearson':means.get('structured_irrelevant',float('nan')),
            'background_mean_lagged_pearson':bm,'proxy_separable':sep,
        }

    @staticmethod
    def _pearson(x, y):
        if len(x)<3 or len(x)!=len(y) or np.std(x)<1e-8 or np.std(y)<1e-8: return 0.
        return float(np.nan_to_num(abs(np.corrcoef(x,y)[0,1]),nan=0.))


print('ProxyLabelQualityChecker ready.')


## 9. Smoke Test: Instance Construction + Diagnostics

In [ ]:
print('='*70)
print('SMOKE TEST: Instance Construction + Diagnostics')
print('='*70)

_builder = TrafficVariableLagDatasetBuilder(SPEED_DF, CFG, ADJ_MATRIX)
_ctx_len = 128
_inst    = None
for _tci in range(min(30, SPEED_DF.shape[1])):
    _inst = _builder.build_instance(_tci, _ctx_len, window_start=0)
    if _inst is not None: break

assert _inst is not None, 'No valid instance found. Check data.'

print(f'  Target sensor: {_inst.target_sensor}  (col {_tci})')
print(f'  T={_inst.T}  context_length={_inst.context_length}  max_lag={_inst.max_lag}')
print(f'  valid_start={_inst.valid_start}  features={len(_inst.features)}')
for m in _inst.metadata:
    print(f'    {m.feature_name[:45]:<45} role={m.feature_role}')
print()
print('-- Leakage checks --')
RealDataLeakageChecks.run_all(_inst)
print()
print('-- Lag variability diagnostics --')
_lag_df = LagVariabilityDiagnostics(_inst.max_lag).compute_for_instance(_inst)
print(_lag_df.groupby('feature_role')[
    ['best_lag_global','lag_variability_score','best_lag_window_unique_count']
].mean().round(3))
print()
print('-- Proxy label quality --')
ProxyLabelQualityChecker(_inst.max_lag).check_instance(_inst, verbose=True)
print()
print('[Smoke test PASSED]')


## 10. Phase 2 Imports

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import average_precision_score, ndcg_score
print('Phase 2 imports OK')


## 11. ClassicalBaselines (Reused)

In [ ]:
class ClassicalBaselines:
    """
    Classical feature-selection baselines for one InstanceData.
    All methods respect valid_start and never use future X values.
    Reused from synthetic notebook with traffic-specific columns added.
    """
    METHODS = ['random','pearson','lagged_pearson','windowed_lagged_pearson',
               'mutual_info','normalized_dtw']

    def __init__(self, cfg): self.cfg = cfg

    @staticmethod
    def _valid_xy(X, Y):
        mask = ~np.isnan(Y)
        return X[mask], Y[mask]

    @staticmethod
    def _pearson_safe(x, y):
        if np.std(x)<1e-8 or np.std(y)<1e-8: return 0.
        return float(np.nan_to_num(abs(np.corrcoef(x,y)[0,1]),nan=0.))

    def _score_random(self, fnames, rng):
        return {n:{'score':float(rng.uniform(0,1)),'best_lag':None,'score_std':None,'valid_windows':None}
                for n in fnames}

    def _score_pearson(self, features, Y):
        out={}
        for n,X in features.items():
            xv,yv=self._valid_xy(X,Y)
            out[n]={'score':self._pearson_safe(xv,yv),'best_lag':0,'score_std':None,'valid_windows':None}
        return out

    def _score_lagged_pearson(self, features, Y, max_lag):
        T=len(Y); vm=~np.isnan(Y); out={}
        for n,X in features.items():
            best,bl=0.,0
            for lag in range(max_lag+1):
                if lag==0: xv,yv=X[vm],Y[vm]
                else: lv=vm[lag:]; xv=X[:T-lag][lv]; yv=Y[lag:][lv]
                if len(xv)<3: continue
                s=self._pearson_safe(xv,yv)
                if s>best: best,bl=s,lag
            out[n]={'score':best,'best_lag':bl,'score_std':None,'valid_windows':None}
        return out

    def _score_windowed_lagged_pearson(self, features, Y, max_lag):
        T=len(Y); vs=int(np.argmax(~np.isnan(Y))); usable=T-vs
        ws=usable//2; wst=max(1,usable//8); out={}
        for n,X in features.items():
            win_scores=[]; bl_win=[]
            for wstart in range(vs,T-ws+1,wst):
                Y_w=Y[wstart:wstart+ws]
                if np.isnan(Y_w).any(): continue
                best,bl=0.,0
                for lag in range(max_lag+1):
                    x0=wstart-lag
                    if x0<0: continue
                    X_w=X[x0:x0+ws]
                    if len(X_w)<ws or np.isnan(X_w).any(): continue
                    s=self._pearson_safe(X_w,Y_w)
                    if s>best: best,bl=s,lag
                win_scores.append(best); bl_win.append(bl)
            if win_scores:
                out[n]={'score':float(np.mean(win_scores)),'best_lag':int(np.mean(bl_win)),
                        'score_std':float(np.std(win_scores)),'valid_windows':len(win_scores)}
            else:
                out[n]={'score':0.,'best_lag':0,'score_std':None,'valid_windows':0}
        return out

    def _score_mutual_info(self, features, Y):
        vm=~np.isnan(Y); yv=Y[vm]; out={}
        for n,X in features.items():
            xv=X[vm].reshape(-1,1)
            try: s=float(mutual_info_regression(xv,yv,random_state=0)[0])
            except: s=0.
            out[n]={'score':s,'best_lag':None,'score_std':None,'valid_windows':None}
        return out

    def _score_normalized_dtw(self, features, Y):
        vm=~np.isnan(Y); yv=Y[vm]; out={}
        for n,X in features.items():
            xv=X[vm]
            if len(xv)<3: out[n]={'score':0.,'best_lag':None,'score_std':None,'valid_windows':None}; continue
            try:
                r,c=len(xv),len(yv)
                D=np.full((r+1,c+1),np.inf); D[0,0]=0.
                for i in range(1,r+1):
                    for j in range(1,c+1):
                        D[i,j]=(xv[i-1]-yv[j-1])**2+min(D[i-1,j],D[i,j-1],D[i-1,j-1])
                s=1./(1.+float(np.sqrt(D[r,c]))/(r+c))
            except: s=0.
            out[n]={'score':s,'best_lag':None,'score_std':None,'valid_windows':None}
        return out

    def compute_all(self, data, random_seed=0):
        rng=np.random.default_rng(random_seed)
        Y=data.Y; ml=data.max_lag
        methods={
            'random':                 self._score_random(list(data.features.keys()),rng),
            'pearson':                self._score_pearson(data.features,Y),
            'lagged_pearson':         self._score_lagged_pearson(data.features,Y,ml),
            'windowed_lagged_pearson':self._score_windowed_lagged_pearson(data.features,Y,ml),
            'mutual_info':            self._score_mutual_info(data.features,Y),
            'normalized_dtw':         self._score_normalized_dtw(data.features,Y),
        }
        rows=[]
        for method_name, scores in methods.items():
            for fn,res in scores.items():
                m=next(x for x in data.metadata if x.feature_name==fn)
                rows.append({
                    'method_name':method_name,'method':method_name,'method_type':'classical',
                    'feature_name':fn,'concept_type':m.concept_type,'feature_role':m.feature_role,
                    'is_causal':m.feature_role in ('causal','proxy_relevant'),
                    'is_matched_distractor':m.is_matched_distractor,'causal_weight':m.causal_weight,
                    'score':res['score'],'raw_score':res['score'],'adjusted_score':None,
                    'score_variant':'raw','score_for_ranking':res['score'],
                    'best_lag':res.get('best_lag'),'score_std':res.get('score_std'),
                    'valid_windows':res.get('valid_windows'),
                    'T':data.T,'seed':data.seed,'noise_level':data.noise_level,
                    'causal_setting':data.causal_setting,'instance_id':data.instance_id,
                    'layer_spec':'classical','pooling_name':'classical',
                    'notebook_name':CFG.NOTEBOOK_NAME,
                    'target_sensor':getattr(data,'target_sensor',''),
                    'context_length':getattr(data,'context_length',data.T),
                    'window_start_idx':getattr(data,'window_start_idx',0),
                    'dataset_name':CFG.DATASET_NAME,
                })
        return pd.DataFrame(rows)


print('ClassicalBaselines ready.')
print(f'  Methods: {ClassicalBaselines.METHODS}')


## 12. Evaluator (Proxy Relevance Labels)

Evaluates feature rankings. `is_causal = (feature_role == 'proxy_relevant')`.
Column `causal_recovery_at_3` = **Proxy Recovery@3** in real-traffic context.
Column `mean_causal_rank` = **Mean Relevant Rank** in real-traffic context.


In [ ]:
class Evaluator:
    """
    Evaluates feature rankings against proxy relevance labels.
    is_causal = (feature_role in ('causal','proxy_relevant'))
    Keeps column names consistent with synthetic notebook for CSV compatibility.
    """
    K = 3

    def evaluate(self, df):
        rows=[]
        cs = df['causal_setting'].iloc[0] if 'causal_setting' in df.columns else 'real_traffic_proxy_labels'
        for method, grp in df.groupby('method_name'):
            labels=grp['is_causal'].astype(int).values
            scores=grp['score'].fillna(-np.inf).values
            m=self._compute_metrics(labels,scores,cs)
            m['method_name']=method; rows.append(m)
        return pd.DataFrame(rows).set_index('method_name')

    def _compute_metrics(self, labels, scores, causal_setting):
        K=self.K
        si=np.argsort(scores)[::-1]; ls=labels[si]; top_k=ls[:K]
        n_c=int(labels.sum())
        try:    ap=float(average_precision_score(labels,scores))
        except: ap=float('nan')
        prec_k   = float(top_k.sum()/K)
        recall_k = float(top_k.sum()/max(n_c,1))
        try:    ndcg_k=float(ndcg_score(labels[np.newaxis,:],scores[np.newaxis,:],k=K))
        except: ndcg_k=float('nan')
        crs=[r+1 for r,lab in enumerate(ls) if lab==1]
        mcr=float(np.mean(crs)) if crs else float('nan')
        return {
            'average_precision':      ap,
            f'precision_at_{K}':      prec_k,
            f'recall_at_{K}':         recall_k,
            f'ndcg_at_{K}':           ndcg_k,
            'mean_causal_rank':       mcr,    # = Mean Relevant Rank for real traffic
            f'causal_recovery_at_{K}': int(top_k.sum()),  # = Proxy Recovery@3
        }


print('Evaluator ready.')
print('  is_causal = feature_role in (causal, proxy_relevant)')
print('  mean_causal_rank    = Mean Relevant Rank (real-traffic plots)')
print('  causal_recovery_at_3 = Proxy Recovery@3 (real-traffic plots)')


## 13. CUDA Check + Chronos Loading

**CUDA is required.** The notebook stops if CUDA is not available.


In [ ]:
import torch
from torch import nn

def print_device_status():
    avail = torch.cuda.is_available()
    print(f'  torch.cuda.is_available() : {avail}')
    if avail:
        print(f'  GPU name                  : {torch.cuda.get_device_name(0)}')
        print(f'  Memory allocated          : {torch.cuda.memory_allocated()/1e9:.3f} GB')
        print(f'  Memory reserved           : {torch.cuda.memory_reserved()/1e9:.3f} GB')
    return avail

print('='*70)
print('CUDA CHECK')
print('='*70)
_cuda_ok = print_device_status()
if not _cuda_ok:
    raise RuntimeError('CUDA is required for this notebook. Run on a machine with a CUDA GPU.')
print('CUDA available. Proceeding.')


In [ ]:
@dataclasses.dataclass
class EmbeddingResult:
    layer_spec:       str
    pooling_name:     str
    embedding_matrix: np.ndarray   # (n_windows, embed_dim)
    window_starts:    List[int]
    window_size:      int
    window_stride:    int
    source_length:    int


class MultiPoolingExtractor:
    """Applies 4 pooling strategies to hidden-state tensor h: (n_windows, seq_len, embed_dim)"""
    POOLING_NAMES = ('mean_pooling','max_pooling','std_pooling','norm_weighted_pooling')

    def pool_all(self, h):
        return {
            'mean_pooling':          h.mean(axis=1),
            'max_pooling':           h.max(axis=1),
            'std_pooling':           h.std(axis=1),
            'norm_weighted_pooling': self._norm_weighted(h),
        }

    @staticmethod
    def _norm_weighted(h):
        norms   = np.linalg.norm(h, axis=-1)
        weights = norms / (norms.sum(axis=1, keepdims=True) + 1e-8)
        return (h * weights[..., np.newaxis]).sum(axis=1)


print('EmbeddingResult and MultiPoolingExtractor ready.')


In [ ]:
try:
    from chronos import BaseChronosPipeline
    print('chronos imported OK')
except ImportError:
    raise ImportError('chronos not found. Activate yael_env: conda activate yael_env')


class ChronosEmbeddingExtractor:
    """
    Loads Chronos-2 and extracts hidden-state representations via forward hooks.

    One forward pass (pipe.embed) per series window batch:
    - All windows are batched and passed to pipe.embed() in a single call
    - All requested layer hidden states are captured from that single pass
    - Single-layer specs, layer-range specs, and all pooling methods are computed
      from the cached hidden states — no rerunning Chronos per layer

    Always call cleanup() after use.
    """

    def __init__(self, cfg):
        self.cfg          = cfg
        self.pipe         = None
        self.model        = None
        self._blocks      = None
        self._hooks: List = []
        self._layer_states: Dict[int, np.ndarray] = {}
        self._pooler      = MultiPoolingExtractor()
        self._n_blocks    = 0

    def load(self):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'  Loading {self.cfg.MODEL_ID} on {device} ...')
        self.pipe  = BaseChronosPipeline.from_pretrained(self.cfg.MODEL_ID, device_map=device)
        self.model = self.pipe.model.eval()
        self._blocks   = self._find_blocks()
        self._n_blocks = len(self._blocks)
        self._register_hooks()
        print(f'  Model loaded: {self._n_blocks} transformer blocks, hooks registered.')
        first_p = next(self.model.parameters(), None)
        if first_p is not None:
            print(f'  First param device: {first_p.device}')
            if str(first_p.device) == 'cpu':
                raise RuntimeError('Model on CPU — CUDA required for this notebook.')

    def cleanup(self):
        for h in self._hooks: h.remove()
        self._hooks.clear()
        self._layer_states.clear()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    def _find_blocks(self):
        hf = self.model.model if hasattr(self.model, 'model') else self.model
        if hasattr(hf, 'layers'): return hf.layers
        if hasattr(hf, 'model') and hasattr(hf.model, 'layers'): return hf.model.layers
        mls = [m for _,m in hf.named_modules() if isinstance(m, nn.ModuleList)]
        if mls: return mls[0]
        raise RuntimeError('Cannot find transformer blocks in the Chronos-2 model.')

    def _register_hooks(self):
        for h in self._hooks: h.remove()
        self._hooks.clear()
        for idx, block in enumerate(self._blocks):
            handle = block.register_forward_hook(self._make_hook(idx))
            self._hooks.append(handle)

    def _make_hook(self, idx):
        def hook(module, input, output):
            if hasattr(output,'hidden_states') and output.hidden_states is not None:
                self._layer_states[idx] = output.hidden_states.detach().cpu().numpy()
            elif isinstance(output, tuple) and len(output) > 0:
                self._layer_states[idx] = output[0].detach().cpu().numpy()
        return hook

    def _forward_pass(self, windows):
        """Single pipe.embed call for all windows — all layer states captured at once."""
        self._layer_states.clear()
        tensors = [torch.tensor(w, dtype=torch.float32) for w in windows]
        with torch.no_grad():
            _ = self.pipe.embed(tensors)

    @staticmethod
    def make_windows(series, window_size, stride):
        T = len(series)
        starts  = list(range(0, T - window_size + 1, stride))
        windows = np.stack([series[s:s+window_size] for s in starts])
        return windows, starts

    def extract_series(
        self,
        series: np.ndarray,
        single_layers: Tuple[int, ...],
        layer_ranges: Tuple[Tuple[int,int], ...],
        window_size: int,
        window_stride: int,
    ) -> Dict[str, Dict[str, EmbeddingResult]]:
        """
        Extract embeddings for all layer specs and pooling strategies.
        One pipe.embed call batches all windows — all hidden states captured at once.

        Args:
            series:        1-D valid array (no NaN)
            single_layers: e.g. (1, 3, 6, 9, 11)
            layer_ranges:  e.g. ((3,6),(6,9),(3,10))
            window_size / window_stride
        Returns:
            dict[layer_spec][pooling_name] -> EmbeddingResult
        """
        assert not np.any(np.isnan(series)), 'series must not contain NaN.'
        windows, starts = self.make_windows(series, window_size, window_stride)
        n_windows       = len(windows)
        source_length   = len(series)

        try:
            # ── Single pipe.embed call — captures ALL layers at once ──────
            self._forward_pass(windows)

            results: Dict[str, Dict[str, EmbeddingResult]] = {}

            # ── Single layers ─────────────────────────────────────────────
            for layer_idx in single_layers:
                if layer_idx not in self._layer_states:
                    raise RuntimeError(
                        f'No hidden state for layer {layer_idx}. '
                        f'Model has {len(self._blocks)} blocks. '
                        f'Captured: {sorted(self._layer_states.keys())}'
                    )
                h    = self._layer_states[layer_idx]   # (n_windows, seq_len, embed_dim)
                spec = f'layer_{layer_idx}'
                results[spec] = {
                    pname: EmbeddingResult(
                        layer_spec=spec, pooling_name=pname, embedding_matrix=mat,
                        window_starts=starts, window_size=window_size,
                        window_stride=window_stride, source_length=source_length,
                    )
                    for pname, mat in self._pooler.pool_all(h).items()
                }

            # ── Layer ranges — pool per layer, then aggregate across layers ─
            for (lo, hi) in layer_ranges:
                range_layers = list(range(lo, hi + 1))
                for agg_name in ('mean', 'max', 'std'):
                    spec = f'range_{lo}_{hi}_{agg_name}'
                    per_layer_pooled: Dict[str, List[np.ndarray]] = {
                        pn: [] for pn in MultiPoolingExtractor.POOLING_NAMES
                    }
                    for layer_idx in range_layers:
                        if layer_idx not in self._layer_states:
                            raise RuntimeError(
                                f'No hidden state for layer {layer_idx} in range ({lo},{hi}).'
                            )
                        h = self._layer_states[layer_idx]
                        for pname, mat in self._pooler.pool_all(h).items():
                            per_layer_pooled[pname].append(mat)
                    results[spec] = {}
                    for pname, mats in per_layer_pooled.items():
                        stacked = np.stack(mats, axis=0)  # (n_range_layers, n_windows, embed_dim)
                        agg = (stacked.mean(0) if agg_name=='mean' else
                               stacked.max(0)  if agg_name=='max'  else stacked.std(0))
                        results[spec][pname] = EmbeddingResult(
                            layer_spec=spec, pooling_name=pname, embedding_matrix=agg,
                            window_starts=starts, window_size=window_size,
                            window_stride=window_stride, source_length=source_length,
                        )
        finally:
            self._layer_states.clear()

        return results


print('ChronosEmbeddingExtractor ready.')
print('  One pipe.embed call batches all windows.')
print('  All layer hidden states captured in a single forward pass.')
print('  No re-running Chronos per layer, per range, or per pooling strategy.')
print('  Missing layer raises RuntimeError (no zero fallback).')


## 14. Extractor Smoke Test (Real METR-LA Data)

Loads Chronos-2, builds 2 real traffic instances, extracts embeddings for
one feature and one target, and prints diagnostic information.

**Run this cell before the full experiment** to verify CUDA, model loading, and embedding extraction.


In [ ]:
print('='*70)
print('EXTRACTOR SMOKE TEST: Real METR-LA data')
print('='*70)

# ── Device status ─────────────────────────────────────────────────────────
print('Device status before loading:')
print_device_status()
print()

# ── Load Chronos ──────────────────────────────────────────────────────────
_extr = ChronosEmbeddingExtractor(CFG)
_extr.load()
print()

# ── Build 2 real instances ─────────────────────────────────────────────────
_smk_builder = TrafficVariableLagDatasetBuilder(SPEED_DF, CFG, ADJ_MATRIX)
_smk_ctx     = 128
_smk_insts   = []
for _stci in range(SPEED_DF.shape[1]):
    _si = _smk_builder.build_instance(_stci, _smk_ctx, window_start=0)
    if _si is not None:
        _smk_insts.append(_si)
    if len(_smk_insts) >= 2: break
assert len(_smk_insts) >= 1, 'No valid instances built from METR-LA.'
print(f'Built {len(_smk_insts)} real instance(s).')
print()

# ── Extract embeddings for target and one feature ─────────────────────────
_si0    = _smk_insts[0]
_vs     = _si0.valid_start
_ctx_l  = _si0.context_length
_ml     = _si0.max_lag
_repr_ws  = _ctx_l // 4
_repr_wst = max(1, _ctx_l // 32)

print(f'Instance: target={_si0.target_sensor}  T={_si0.T}  max_lag={_ml}')
print(f'Repr window: {_repr_ws}  stride: {_repr_wst}')
print()

# Extract Y (target, valid portion only)
_Y_valid = _si0.Y[_vs:]
assert not np.any(np.isnan(_Y_valid)), 'Y has NaN in valid region'
print('Extracting Y embeddings ...')
import time as _t
_t0 = _t.perf_counter()
_Y_embs = _extr.extract_series(
    _Y_valid, CFG.SINGLE_LAYERS, CFG.LAYER_RANGES, _repr_ws, _repr_wst
)
_elapsed = _t.perf_counter() - _t0
print(f'  Done in {_elapsed:.2f}s')
print()

# Extract X (first feature)
_fn0, _X_full = list(_si0.features.items())[0]
_X_valid = _X_full[_vs:]
assert not np.any(np.isnan(_X_valid)), f'Feature {_fn0} has NaN in valid region'
print(f'Extracting X embeddings for {_fn0} ...')
_t0 = _t.perf_counter()
_X_embs = _extr.extract_series(
    _X_valid, CFG.SINGLE_LAYERS, CFG.LAYER_RANGES, _repr_ws, _repr_wst
)
_elapsed = _t.perf_counter() - _t0
print(f'  Done in {_elapsed:.2f}s')
print()

# ── Diagnostics ──────────────────────────────────────────────────────────
print('── EXTRACTOR DIAGNOSTICS ─────────────────────────────')
_sample_spec = list(_Y_embs.keys())[0]
_sample_pn   = list(_Y_embs[_sample_spec].keys())[0]
_sample_emb  = _Y_embs[_sample_spec][_sample_pn]
print(f'  n_windows (Y)                  : {_sample_emb.embedding_matrix.shape[0]}')
print(f'  embed_dim                      : {_sample_emb.embedding_matrix.shape[1]}')
print(f'  window_size / stride           : {_sample_emb.window_size} / {_sample_emb.window_stride}')
print(f'  Number of layer specs          : {len(_Y_embs)}')
print(f'  Layer specs found              : {sorted(_Y_embs.keys())}')
print(f'  Pooling methods found          : {sorted(_Y_embs[_sample_spec].keys())}')
print(f'  Chronos embed calls for Y      : 1  (all windows batched)')
print(f'  Chronos embed calls for X      : 1  (all windows batched)')
print(f'  Total Chronos calls so far     : 2  (Y + 1 feature)')
print(f'  Model device                   : {next(_extr.model.parameters()).device}')
print()
print('Device status after embedding:')
print_device_status()
print()
print('[Extractor smoke test PASSED]')
print()
print('Extractor stored in _extr for use in ExperimentRunner.')
print('Call _extr.cleanup() after the run to free GPU memory.')


## 15. RepresentationMetrics (+ Centered Cosine Variants)

Extends the original metrics with three centered cosine methods:
- `centered_cosine`: subtract instance-level grand mean before cosine
- `windowed_centered_cosine`: per-window centered cosine
- `lagged_centered_cosine`: center then compute lagged cosine

Grand mean = mean embedding across ALL features + target for this instance,
within each (layer_spec, pooling_name). Centering scope: `instance_feature_target_mean`.


In [ ]:
class RepresentationMetrics:
    """
    Stateless class computing representation similarity metrics.
    Reused from synthetic notebook + centered cosine variants added.

    Base metrics: cka, dcor, cosine, l2, windowed_cka, windowed_cosine
    Lagged metrics: lagged_cka, windowed_lagged_cka, lagged_cosine
    Centered variants: centered_cosine, windowed_centered_cosine, lagged_centered_cosine
    """

    METHODS = [
        'cka','dcor','cosine','l2','windowed_cka','windowed_cosine',
        'lagged_cka','windowed_lagged_cka','lagged_cosine',
        'centered_cosine','windowed_centered_cosine','lagged_centered_cosine',
    ]

    @staticmethod
    def _validate_windows(n, name=''):
        if n < 5:
            if name: print(f'  [WARN] {name}: n_windows={n} < 5 — returning NaN.')
            return {'_ok':False,'research_eligible':False,'diagnostic_only_low_window_count':True}
        return {'_ok':True,'research_eligible':n>=10,'diagnostic_only_low_window_count':n<10}

    @staticmethod
    def _info(d): return {k:v for k,v in d.items() if k!='_ok'}

    # ── CKA ──────────────────────────────────────────────────────────────
    @staticmethod
    def _cka_core(Hx, Hy):
        Hxc=Hx-Hx.mean(0); Hyc=Hy-Hy.mean(0)
        cross=Hxc.T@Hyc
        num=np.linalg.norm(cross,'fro')**2
        denom=np.linalg.norm(Hxc.T@Hxc,'fro')*np.linalg.norm(Hyc.T@Hyc,'fro')
        return float(num/denom) if denom>1e-10 else 0.

    @classmethod
    def cka(cls, Hx, Hy):
        n=Hx.shape[0]; info=cls._validate_windows(n,'cka')
        return {'score':cls._cka_core(Hx,Hy) if info['_ok'] else float('nan'),
                'n_windows':n,**cls._info(info)}

    @classmethod
    def windowed_cka(cls, Hx, Hy):
        r=cls.cka(Hx,Hy); r['method_note']='windowed_cka==cka in this phase'; return r

    # ── dCor ─────────────────────────────────────────────────────────────
    @staticmethod
    def _dcor_manual(Hx, Hy):
        def pw(H):
            sq=np.sum(H**2,axis=1,keepdims=True)
            return np.sqrt(np.maximum(sq+sq.T-2.*(H@H.T),0.))
        def dc(D): return D-D.mean(1,keepdims=True)-D.mean(0,keepdims=True)+D.mean()
        A,B=dc(pw(Hx)),dc(pw(Hy))
        d2=np.sum(A*A)*np.sum(B*B)
        return float(np.sqrt(max(np.sum(A*B)/np.sqrt(d2),0.))) if d2>0 else 0.

    @classmethod
    def dcor(cls, Hx, Hy, allow_fallback=True):
        n=Hx.shape[0]; info=cls._validate_windows(n,'dcor')
        try:
            import dcor as _dp
            s=float(_dp.distance_correlation(Hx,Hy)) if info['_ok'] else float('nan')
            return {'score':s,'n_windows':n,'dcor_status':'package',
                    'dcor_implementation':'official_dcor_package',**cls._info(info)}
        except ImportError:
            if not allow_fallback:
                return {'score':float('nan'),'n_windows':n,'dcor_status':'unavailable',
                        'dcor_implementation':None,**cls._info(info)}
            s=cls._dcor_manual(Hx,Hy) if info['_ok'] else float('nan')
            return {'score':s,'n_windows':n,'dcor_status':'fallback',
                    'dcor_implementation':'manual_distance_correlation_double_centering',
                    'research_eligible':False,'diagnostic_only_low_window_count':True,
                    **{k:v for k,v in cls._info(info).items() if k not in ('research_eligible','diagnostic_only_low_window_count')}}

    # ── Cosine ────────────────────────────────────────────────────────────
    @classmethod
    def cosine(cls, Hx, Hy):
        n=Hx.shape[0]; info=cls._validate_windows(n,'cosine')
        vx,vy=Hx.mean(0),Hy.mean(0)
        nx,ny=np.linalg.norm(vx),np.linalg.norm(vy)
        s=float(np.dot(vx,vy)/(nx*ny)) if nx>1e-10 and ny>1e-10 else 0.
        return {'score':s,'n_windows':n,**cls._info(info)}

    @classmethod
    def windowed_cosine(cls, Hx, Hy):
        n=Hx.shape[0]; info=cls._validate_windows(n,'windowed_cosine')
        nx=np.linalg.norm(Hx,axis=1,keepdims=True); ny=np.linalg.norm(Hy,axis=1,keepdims=True)
        Hxn=Hx/np.maximum(nx,1e-10); Hyn=Hy/np.maximum(ny,1e-10)
        pw=np.sum(Hxn*Hyn,axis=1)
        return {'score':float(pw.mean()),'score_std':float(pw.std()),
                'n_windows':n,**cls._info(info)}

    # ── L2 ────────────────────────────────────────────────────────────────
    @classmethod
    def l2(cls, Hx, Hy):
        n=Hx.shape[0]; info=cls._validate_windows(n,'l2')
        vx,vy=Hx.mean(0),Hy.mean(0)
        nx,ny=np.linalg.norm(vx),np.linalg.norm(vy)
        vxn=vx/nx if nx>1e-10 else vx; vyn=vy/ny if ny>1e-10 else vy
        dist=float(np.linalg.norm(vxn-vyn)); sim=1./(1.+dist)
        return {'score':sim,'l2_distance':dist,'l2_similarity':sim,
                'n_windows':n,**cls._info(info)}

    # ── Centered cosine ───────────────────────────────────────────────────
    @classmethod
    def centered_cosine(cls, Hx, Hy, shared_mean=None):
        """
        Centered cosine: subtract shared mean embedding before cosine.
        shared_mean: grand mean across all features+target for this instance/layer/pooling.
        centering_scope: 'instance_feature_target_mean'
        """
        n=Hx.shape[0]; info=cls._validate_windows(n,'centered_cosine')
        if shared_mean is None:
            shared_mean = np.vstack([Hx,Hy]).mean(0)
        Hxc=Hx-shared_mean; Hyc=Hy-shared_mean
        vx,vy=Hxc.mean(0),Hyc.mean(0)
        nx,ny=np.linalg.norm(vx),np.linalg.norm(vy)
        s=float(np.dot(vx,vy)/(nx*ny)) if nx>1e-10 and ny>1e-10 else 0.
        return {'score':s,'n_windows':n,'centering_scope':'instance_feature_target_mean',
                **cls._info(info)}

    @classmethod
    def windowed_centered_cosine(cls, Hx, Hy, shared_mean=None):
        """
        Per-window centered cosine: subtract shared_mean, then compute per-window cosine.
        """
        n=Hx.shape[0]; info=cls._validate_windows(n,'windowed_centered_cosine')
        if shared_mean is None:
            shared_mean = np.vstack([Hx,Hy]).mean(0)
        Hxc=Hx-shared_mean; Hyc=Hy-shared_mean
        nx=np.linalg.norm(Hxc,axis=1,keepdims=True)
        ny=np.linalg.norm(Hyc,axis=1,keepdims=True)
        Hxcn=Hxc/np.maximum(nx,1e-10)
        Hycn=Hyc/np.maximum(ny,1e-10)
        pw=np.sum(Hxcn*Hycn,axis=1)
        return {'score':float(pw.mean()),'score_std':float(pw.std()),
                'n_windows':n,'centering_scope':'instance_feature_target_mean',
                **cls._info(info)}

    # ── _extract_lag: shift raw series before embedding (never shift embeddings) ──
    @classmethod
    def _extract_lag(cls, extractor, X_full, valid_start, T, lag,
                     repr_ws, repr_wstr, single_layers, layer_ranges, Y=None):
        _val_fn = globals().get('validate_lagged_alignment_indices')
        if _val_fn is not None:
            _val = _val_fn(valid_start, T, lag, X=X_full, Y=Y)
            if not _val['is_valid']:
                print(f'  [ERROR] Lag {lag}: {_val["error_message"]}')
                return None
        X_aligned = X_full[valid_start-lag : T-lag]
        try:
            return extractor.extract_series(X_aligned, single_layers, layer_ranges,
                                            repr_ws, repr_wstr)
        except Exception as e:
            print(f'  [WARN] extract_series failed lag={lag}: {e}')
            return None

    # ── Lagged CKA ────────────────────────────────────────────────────────
    @classmethod
    def lagged_cka(cls, extractor, X_full, Y_embs_cached, data,
                   layer_spec, pooling_name, repr_ws, repr_wstr,
                   single_layers, layer_ranges):
        vs,T,ml=data.valid_start,data.T,data.max_lag
        Hy=Y_embs_cached[layer_spec][pooling_name].embedding_matrix
        best,blg,bnw=float('-inf'),0,0; all_scores=[]
        for lag in range(ml+1):
            x_embs=cls._extract_lag(extractor,X_full,vs,T,lag,repr_ws,repr_wstr,
                                    single_layers,layer_ranges,Y=data.Y)
            if x_embs is None: all_scores.append(float('nan')); continue
            Hx=x_embs[layer_spec][pooling_name].embedding_matrix
            r=cls.cka(Hx,Hy); s=r['score']; all_scores.append(s)
            if not np.isnan(s) and s>best: best,blg,bnw=s,lag,r['n_windows']
        info=cls._validate_windows(bnw,'lagged_cka')
        return {'score':best if not np.isinf(best) else float('nan'),
                'best_lag':blg,'n_windows_at_best_lag':bnw,'all_lag_scores':all_scores,
                'n_windows':bnw,**cls._info(info)}

    @classmethod
    def windowed_lagged_cka(cls, extractor, X_full, Y_embs_cached, data,
                            layer_spec, pooling_name, repr_ws, repr_wstr,
                            single_layers, layer_ranges):
        r=cls.lagged_cka(extractor,X_full,Y_embs_cached,data,layer_spec,pooling_name,
                         repr_ws,repr_wstr,single_layers,layer_ranges)
        r['method_note']='windowed_lagged_cka==lagged_cka in this phase'
        return r

    # ── Lagged cosine ─────────────────────────────────────────────────────
    @classmethod
    def lagged_cosine(cls, extractor, X_full, Y_embs_cached, data,
                      layer_spec, pooling_name, repr_ws, repr_wstr,
                      single_layers, layer_ranges):
        vs,T,ml=data.valid_start,data.T,data.max_lag
        Hy=Y_embs_cached[layer_spec][pooling_name].embedding_matrix
        best,blg,bnw=float('-inf'),0,0; all_scores=[]
        for lag in range(ml+1):
            x_embs=cls._extract_lag(extractor,X_full,vs,T,lag,repr_ws,repr_wstr,
                                    single_layers,layer_ranges,Y=data.Y)
            if x_embs is None: all_scores.append(float('nan')); continue
            Hx=x_embs[layer_spec][pooling_name].embedding_matrix
            r=cls.cosine(Hx,Hy); s=r['score']; all_scores.append(s)
            if not np.isnan(s) and s>best: best,blg,bnw=s,lag,r['n_windows']
        info=cls._validate_windows(bnw,'lagged_cosine')
        return {'score':best if not np.isinf(best) else float('nan'),
                'best_lag':blg,'n_windows_at_best_lag':bnw,'all_lag_scores':all_scores,
                'n_windows':bnw,**cls._info(info)}

    # ── Lagged centered cosine ────────────────────────────────────────────
    @classmethod
    def lagged_centered_cosine(cls, extractor, X_full, Y_embs_cached, data,
                               layer_spec, pooling_name, repr_ws, repr_wstr,
                               single_layers, layer_ranges, shared_mean=None):
        """
        Shifts raw X before embedding (never shifts embedding rows).
        Compares X[t-lag] with Y[t]. Applies centering before cosine.
        """
        vs,T,ml=data.valid_start,data.T,data.max_lag
        Hy=Y_embs_cached[layer_spec][pooling_name].embedding_matrix
        best,blg,bnw=float('-inf'),0,0; all_scores=[]
        for lag in range(ml+1):
            x_embs=cls._extract_lag(extractor,X_full,vs,T,lag,repr_ws,repr_wstr,
                                    single_layers,layer_ranges,Y=data.Y)
            if x_embs is None: all_scores.append(float('nan')); continue
            Hx=x_embs[layer_spec][pooling_name].embedding_matrix
            r=cls.centered_cosine(Hx,Hy,shared_mean=shared_mean)
            s=r['score']; all_scores.append(s)
            if not np.isnan(s) and s>best: best,blg,bnw=s,lag,r['n_windows']
        info=cls._validate_windows(bnw,'lagged_centered_cosine')
        return {'score':best if not np.isinf(best) else float('nan'),
                'best_lag':blg,'n_windows_at_best_lag':bnw,'all_lag_scores':all_scores,
                'n_windows':bnw,'centering_scope':'instance_feature_target_mean',
                **cls._info(info)}


print('RepresentationMetrics ready.')
print('  Base: cka, dcor, cosine, l2, windowed_cka, windowed_cosine')
print('  Lagged: lagged_cka, windowed_lagged_cka, lagged_cosine')
print('  Centered: centered_cosine, windowed_centered_cosine, lagged_centered_cosine')
print('  Grand mean for centering: all features + target, per (layer_spec, pooling_name)')


In [ ]:
class NegativeControlAdjustment:
    """
    Computes negative-control adjusted scores.
    adjusted_score = raw_score - mean(background_irrelevant_scores)
    """
    CONTROL_TYPES = ('background_irrelevant',)

    @staticmethod
    def adjust_background(feature_scores, metadata):
        bg_names  = {m.feature_name for m in metadata if m.feature_role=='background_irrelevant'}
        bg_scores = [v for k,v in feature_scores.items()
                     if k in bg_names and not np.isnan(float(v or float('nan')))]
        cm = float(np.mean(bg_scores)) if bg_scores else 0.
        cs = float(np.std(bg_scores))  if bg_scores else 0.
        return {
            feat: {'raw_score':score,
                   'adjusted_score': score-cm if not np.isnan(float(score or float('nan'))) else float('nan'),
                   'negative_control_type':'background_irrelevant',
                   'negative_control_mean':cm,'negative_control_std':cs}
            for feat,score in feature_scores.items()
        }


print('NegativeControlAdjustment ready.')


## 16. Embedding Cache

In-memory (+ optional disk) cache for Chronos embeddings.
Prevents re-running Chronos for the same (series_hash, layer_spec, pooling_name, window_size, stride).


In [ ]:
class EmbeddingCache:
    """
    Cache for EmbeddingResult objects.

    Cache key: (series_hash, layer_spec, pooling_name, window_size, window_stride)
    where series_hash = sha1 of the raw series bytes.

    Optional disk cache: saves/loads pickle files to cache_dir.
    """

    def __init__(self, cache_dir=None):
        self._mem: Dict[tuple, EmbeddingResult] = {}
        self._disk = pathlib.Path(cache_dir) if cache_dir else None
        if self._disk: self._disk.mkdir(parents=True, exist_ok=True)
        self.hits   = 0
        self.misses = 0
        self._total_embed_calls = 0

    @staticmethod
    def _series_hash(series: np.ndarray) -> str:
        return hashlib.sha1(series.tobytes()).hexdigest()[:16]

    def key(self, series, layer_spec, pooling_name, window_size, window_stride) -> tuple:
        return (self._series_hash(series), layer_spec, pooling_name, window_size, window_stride)

    def get(self, key: tuple) -> Optional[EmbeddingResult]:
        if key in self._mem:
            self.hits += 1
            return self._mem[key]
        if self._disk:
            p = self._disk / (hashlib.sha1(str(key).encode()).hexdigest() + '.pkl')
            if p.exists():
                try:
                    with open(p,'rb') as f: r = pickle.load(f)
                    self._mem[key] = r; self.hits += 1; return r
                except: pass
        self.misses += 1
        return None

    def put(self, key: tuple, value: EmbeddingResult):
        self._mem[key] = value
        if self._disk:
            p = self._disk / (hashlib.sha1(str(key).encode()).hexdigest() + '.pkl')
            try:
                with open(p,'wb') as f: pickle.dump(value, f)
            except: pass

    def register_embed_call(self): self._total_embed_calls += 1

    def stats(self):
        total = self.hits + self.misses
        hr = self.hits/total if total>0 else 0.
        return {
            'cache_hits':   self.hits,
            'cache_misses': self.misses,
            'hit_rate':     hr,
            'total_embed_calls': self._total_embed_calls,
        }

    def print_stats(self):
        s = self.stats()
        print(f'  Cache hits      : {s["cache_hits"]}')
        print(f'  Cache misses    : {s["cache_misses"]}')
        print(f'  Hit rate        : {s["hit_rate"]*100:.1f}%')
        print(f'  Chronos calls   : {s["total_embed_calls"]}')


print('EmbeddingCache ready.')


In [ ]:
class RepresentationScoreRunner:
    """
    Orchestrates Chronos embedding extraction and representation metric computation
    for one InstanceData. Adapted from the synthetic notebook:
    - Collects all X embeddings first to compute grand mean for centered cosine
    - Uses EmbeddingCache to avoid redundant Chronos calls
    - is_causal = feature_role in ('causal', 'proxy_relevant')
    """

    BASE_METRICS    = ['cka','dcor','cosine','l2','windowed_cka','windowed_cosine']
    LAGGED_METRICS  = ['lagged_cka','windowed_lagged_cka','lagged_cosine']
    CENTERED_METRICS= ['centered_cosine','windowed_centered_cosine','lagged_centered_cosine']
    _WINDOWED       = {'windowed_cka','windowed_cosine','windowed_lagged_cka',
                       'windowed_centered_cosine'}

    def __init__(self, cfg, extractor, single_layers=None, layer_ranges=None, cache=None):
        self.cfg           = cfg
        self.extractor     = extractor
        self.single_layers = single_layers if single_layers is not None else cfg.SINGLE_LAYERS
        self.layer_ranges  = layer_ranges  if layer_ranges  is not None else cfg.LAYER_RANGES
        self.cache         = cache  # EmbeddingCache or None
        self.RM            = RepresentationMetrics

    def _extract_with_cache(self, series, repr_ws, repr_wstr):
        """Extract embeddings with in-memory cache."""
        series_hash = hashlib.sha1(series.tobytes()).hexdigest()[:16]
        if self.cache is not None:
            # Try to get all specs from cache
            all_specs   = ([f'layer_{li}' for li in self.single_layers] +
                           [f'range_{lo}_{hi}_{ag}'
                            for lo,hi in self.layer_ranges
                            for ag in ('mean','max','std')])
            all_poolings = MultiPoolingExtractor.POOLING_NAMES
            results = {}
            hit_all = True
            for spec in all_specs:
                for pn in all_poolings:
                    k = (series_hash, spec, pn, repr_ws, repr_wstr)
                    r = self.cache.get(k)
                    if r is None: hit_all = False; break
                    if spec not in results: results[spec] = {}
                    results[spec][pn] = r
                if not hit_all: break
            if hit_all and results:
                return results
        # Cache miss — call extractor
        if self.cache: self.cache.register_embed_call()
        results = self.extractor.extract_series(
            series, self.single_layers, self.layer_ranges, repr_ws, repr_wstr
        )
        if self.cache:
            for spec, pn_dict in results.items():
                for pn, emb in pn_dict.items():
                    k = (series_hash, spec, pn, repr_ws, repr_wstr)
                    self.cache.put(k, emb)
        return results

    def compute_all(
        self, data, repr_ws, repr_wstr, layer_specs, pooling_names,
        metrics_to_run=None, run_lagged=True,
        lagged_layer_spec='layer_6', lagged_pooling='mean_pooling',
        apply_neg_control=True, allow_dcor_fallback=True,
    ) -> pd.DataFrame:

        if metrics_to_run is None: metrics_to_run = RepresentationMetrics.METHODS
        meta_map = {m.feature_name: m for m in data.metadata}
        vs = data.valid_start

        # ── Extract Y embeddings ──────────────────────────────────────────
        Y_valid = data.Y[vs:]
        Y_embs  = self._extract_with_cache(Y_valid, repr_ws, repr_wstr)

        # ── Extract ALL X embeddings first (needed for grand mean) ────────
        all_X_embs: Dict[str, dict] = {}
        for feat_name, X_full in data.features.items():
            all_X_embs[feat_name] = self._extract_with_cache(X_full[vs:], repr_ws, repr_wstr)

        # ── Compute grand mean per (layer_spec, pooling_name) for centered cosine ─
        needs_centering = any(m in metrics_to_run for m in self.CENTERED_METRICS)
        grand_means: Dict[tuple, np.ndarray] = {}
        if needs_centering:
            for ls in layer_specs:
                for pn in pooling_names:
                    mats = []
                    for fn in data.features:
                        if ls in all_X_embs[fn] and pn in all_X_embs[fn][ls]:
                            mats.append(all_X_embs[fn][ls][pn].embedding_matrix)
                    if ls in Y_embs and pn in Y_embs[ls]:
                        mats.append(Y_embs[ls][pn].embedding_matrix)
                    if mats:
                        grand_means[(ls,pn)] = np.vstack(mats).mean(0)

        rows = []
        for feat_name, X_full in data.features.items():
            m       = meta_map[feat_name]
            X_embs  = all_X_embs[feat_name]

            for ls in layer_specs:
                if ls not in X_embs or ls not in Y_embs: continue
                for pn in pooling_names:
                    if pn not in X_embs[ls] or pn not in Y_embs[ls]: continue
                    Hx    = X_embs[ls][pn].embedding_matrix
                    Hy    = Y_embs[ls][pn].embedding_matrix
                    e_res = X_embs[ls][pn]
                    gm    = grand_means.get((ls,pn))

                    # Base metrics
                    _base_dispatch = {
                        'cka':             self.RM.cka,
                        'cosine':          self.RM.cosine,
                        'l2':              self.RM.l2,
                        'windowed_cka':    self.RM.windowed_cka,
                        'windowed_cosine': self.RM.windowed_cosine,
                    }
                    for mn, fn_m in _base_dispatch.items():
                        if mn not in metrics_to_run: continue
                        res  = fn_m(Hx, Hy)
                        mtyp = 'representation_windowed' if mn in self._WINDOWED else 'representation_base'
                        row  = self._row(data, m, mn, ls, pn, res, e_res, repr_ws, repr_wstr, mtyp)
                        if row: rows.append(row)

                    # dCor
                    if 'dcor' in metrics_to_run:
                        res = self.RM.dcor(Hx, Hy, allow_fallback=allow_dcor_fallback)
                        row = self._row(data, m, 'dcor', ls, pn, res, e_res, repr_ws, repr_wstr, 'representation_base')
                        if row: rows.append(row)

                    # Centered cosine variants
                    if 'centered_cosine' in metrics_to_run:
                        res  = self.RM.centered_cosine(Hx, Hy, shared_mean=gm)
                        row  = self._row(data, m, 'centered_cosine', ls, pn, res, e_res, repr_ws, repr_wstr, 'representation_base')
                        if row: rows.append(row)
                    if 'windowed_centered_cosine' in metrics_to_run:
                        res  = self.RM.windowed_centered_cosine(Hx, Hy, shared_mean=gm)
                        row  = self._row(data, m, 'windowed_centered_cosine', ls, pn, res, e_res, repr_ws, repr_wstr, 'representation_windowed')
                        if row: rows.append(row)

            # Lagged metrics (one layer_spec/pooling)
            if run_lagged:
                ls_l, pn_l = lagged_layer_spec, lagged_pooling
                if ls_l not in Y_embs or pn_l not in Y_embs.get(ls_l,{}): continue
                gm_l = grand_means.get((ls_l, pn_l))
                for lag_mn in self.LAGGED_METRICS:
                    if lag_mn not in metrics_to_run: continue
                    fn_l = getattr(self.RM, lag_mn)
                    lag_r = fn_l(self.extractor, X_full, Y_embs, data, ls_l, pn_l,
                                 repr_ws, repr_wstr, self.single_layers, self.layer_ranges)
                    e_res = Y_embs[ls_l][pn_l]
                    mtyp  = 'representation_windowed_lagged' if lag_mn in self._WINDOWED else 'representation_lagged'
                    row   = self._row(data, m, lag_mn, ls_l, pn_l, lag_r, e_res, repr_ws, repr_wstr, mtyp)
                    if row: rows.append(row)
                if 'lagged_centered_cosine' in metrics_to_run:
                    lag_r = self.RM.lagged_centered_cosine(
                        self.extractor, X_full, Y_embs, data, ls_l, pn_l,
                        repr_ws, repr_wstr, self.single_layers, self.layer_ranges,
                        shared_mean=gm_l
                    )
                    e_res = Y_embs[ls_l][pn_l]
                    row   = self._row(data, m, 'lagged_centered_cosine', ls_l, pn_l, lag_r, e_res, repr_ws, repr_wstr, 'representation_lagged')
                    if row: rows.append(row)

        if not rows: return pd.DataFrame()
        df = pd.DataFrame(rows)
        if apply_neg_control: df = self._apply_neg_control(df, data.metadata)
        return self._rank(df)

    def _row(self, data, meta, method_name, ls, pn, res, emb_ref,
             repr_ws, repr_wstr, method_type):
        if res is None: return None
        dcor_status = res.get('dcor_status')
        if method_name=='dcor' and dcor_status=='unavailable': return None
        eff_mn = 'dcor_fallback' if (method_name=='dcor' and dcor_status=='fallback') else method_name
        n_win  = res.get('n_windows', emb_ref.embedding_matrix.shape[0])
        raw    = res.get('score', float('nan'))
        all_lag_raw = res.get('all_lag_scores')
        all_lag_str = _json.dumps([None if (isinstance(s,float) and np.isnan(s)) else s
                                   for s in all_lag_raw]) if all_lag_raw else None
        # Rolling validation for windowed methods
        if eff_mn in self._WINDOWED:
            _RWV = globals().get('RollingWindowValidator')
            if _RWV:
                sl   = getattr(emb_ref,'source_length', n_win*repr_wstr+repr_ws)
                _rv  = _RWV(repr_ws,repr_wstr,sl).validate_config()
                rstat= 'valid' if _rv['is_valid'] else 'invalid'
                rerrs= '; '.join(_rv['errors']) if _rv['errors'] else None
            else: rstat,rerrs = 'not_validated',None
        else: rstat,rerrs = None,None

        return {
            'timestamp': None,
            'notebook_name':   self.cfg.NOTEBOOK_NAME,
            'dataset_name':    self.cfg.DATASET_NAME,
            'target_sensor':   getattr(data,'target_sensor',''),
            'context_length':  getattr(data,'context_length',data.T),
            'window_start_idx':getattr(data,'window_start_idx',0),
            'seed':data.seed,'T':data.T,'noise_level':data.noise_level,
            'causal_setting':data.causal_setting,'instance_id':data.instance_id,
            'feature_name':meta.feature_name,'concept_type':meta.concept_type,
            'feature_role':meta.feature_role,
            'is_causal': meta.feature_role in ('causal','proxy_relevant'),
            'is_matched_distractor':meta.is_matched_distractor,'causal_weight':meta.causal_weight,
            'method_name':eff_mn,'method':eff_mn,'method_type':method_type,
            'layer_spec':ls,'pooling_name':pn,
            'score':raw,'raw_score':raw,'adjusted_score':None,
            'score_variant':'raw','score_for_ranking':raw,
            'negative_control_type':None,'negative_control_mean':None,'negative_control_std':None,
            'centering_scope':res.get('centering_scope'),
            'best_lag':res.get('best_lag'),'n_windows_at_best_lag':res.get('n_windows_at_best_lag'),
            'all_lag_scores':all_lag_str,'score_std':res.get('score_std'),
            'l2_distance':res.get('l2_distance'),'l2_similarity':res.get('l2_similarity'),
            'n_windows':n_win,
            'representation_window_size':repr_ws,'representation_window_stride':repr_wstr,
            'diagnostic_only_low_window_count':res.get('diagnostic_only_low_window_count',n_win<10),
            'research_eligible':res.get('research_eligible',n_win>=10),
            'dcor_status':dcor_status,'dcor_implementation':res.get('dcor_implementation'),
            'rolling_validation_status':rstat,'rolling_validation_errors':rerrs,
            'window_size_used':repr_ws,'window_stride_used':repr_wstr,'window_type':'representation',
            'method_note':res.get('method_note'),'rank':None,'runtime_seconds':None,
        }

    def _apply_neg_control(self, df, metadata):
        df = df.copy()
        for (mn,ls,pn), grp in df.groupby(['method_name','layer_spec','pooling_name']):
            sc = dict(zip(grp['feature_name'], grp['raw_score'].fillna(float('nan'))))
            adj= NegativeControlAdjustment.adjust_background(sc, metadata)
            for feat, info in adj.items():
                mask = ((df['method_name']==mn)&(df['layer_spec']==ls)
                        &(df['pooling_name']==pn)&(df['feature_name']==feat))
                df.loc[mask,'adjusted_score']        = info['adjusted_score']
                df.loc[mask,'negative_control_type'] = info['negative_control_type']
                df.loc[mask,'negative_control_mean'] = info['negative_control_mean']
                df.loc[mask,'negative_control_std']  = info['negative_control_std']
        return df

    @staticmethod
    def _rank(df):
        df=df.copy()
        for _,(mn,ls,pn) in df.groupby(['method_name','layer_spec','pooling_name']).groups.items():
            pass
        for _, grp in df.groupby(['method_name','layer_spec','pooling_name']):
            df.loc[grp.index,'rank']=grp['raw_score'].fillna(-np.inf).rank(
                ascending=False,method='min').astype(int)
        return df


print('RepresentationScoreRunner ready.')
print('  Centered cosine grand mean: all features+target per (layer_spec, pooling_name).')
print('  EmbeddingCache: one Chronos call per unique series.')
print('  is_causal = feature_role in (causal, proxy_relevant).')


## 17. dCor Availability Check

In [ ]:
try:
    import dcor as _dcor_pkg
    _DCOR_AVAILABLE      = True
    _DCOR_STATUS_LABEL   = 'package'
    _DCOR_IMPLEMENTATION = 'official_dcor_package'
    print("dCor: official 'dcor' package available.")
except ImportError:
    _dcor_pkg            = None
    _DCOR_AVAILABLE      = False
    _DCOR_STATUS_LABEL   = 'fallback'
    _DCOR_IMPLEMENTATION = 'manual_distance_correlation_double_centering'
    print("dCor: 'dcor' package NOT installed.")
    print('  Diagnostic: method_name=dcor_fallback, research_eligible=False')
    print('  Core/research: dCor row SKIPPED entirely (no silent fallback)')
print(f'  _DCOR_AVAILABLE={_DCOR_AVAILABLE}  label={_DCOR_STATUS_LABEL!r}')


## 18. Validation Helpers + Score Variant Utilities (Reused)

In [ ]:
def validate_lagged_alignment_indices(
        valid_start, T, lag, X=None, Y=None):
    """
    Validates lagged alignment.
    X_aligned = X_full[valid_start-lag : T-lag]   (length = T - valid_start)
    Y_aligned = Y[valid_start : T]                (length = T - valid_start)
    Never shifts embedding rows after embedding. Never uses future values.
    """
    x_start        = valid_start - lag
    x_end          = T - lag
    y_start        = valid_start
    y_end          = T
    aligned_length = x_end - x_start
    errors = []
    if lag < 0: errors.append(f'lag={lag} is negative.')
    if x_start < 0: errors.append(f'x_start={x_start} < 0: valid_start={valid_start} < lag={lag}.')
    if x_end > T:   errors.append(f'x_end={x_end} > T={T}.')
    if aligned_length != (y_end-y_start): errors.append('Length mismatch.')
    if aligned_length <= 0: errors.append(f'aligned_length={aligned_length} <= 0.')
    x_nan_checked = y_nan_checked = False
    if X is not None and not errors:
        xs = X[x_start:x_end]
        if len(xs)!=aligned_length: errors.append(f'X slice len={len(xs)} != {aligned_length}.')
        elif np.isnan(xs).any(): errors.append(f'NaN in X[{x_start}:{x_end}].')
        x_nan_checked = True
    if Y is not None and not errors:
        ys = Y[y_start:y_end]
        if len(ys)!=aligned_length: errors.append(f'Y slice len={len(ys)} != {aligned_length}.')
        elif np.isnan(ys).any(): errors.append(f'NaN in Y[{y_start}:{y_end}].')
        y_nan_checked = True
    return {'lag':lag,'x_start':x_start,'x_end':x_end,'y_start':y_start,'y_end':y_end,
            'aligned_length':aligned_length,'x_nan_checked':x_nan_checked,
            'y_nan_checked':y_nan_checked,'is_valid':not errors,
            'error_message':'; '.join(errors) if errors else None}


class RollingWindowValidator:
    """Validates rolling-window configuration for any windowed method."""
    def __init__(self, window_size, stride, series_length):
        self.window_size=window_size; self.stride=stride; self.series_length=series_length
    def validate_config(self):
        ws,wst,L=self.window_size,self.stride,self.series_length
        errors=[]
        if ws<=0: errors.append(f'window_size={ws} must be > 0.')
        if wst<=0: errors.append(f'stride={wst} must be > 0.')
        if ws>L: errors.append(f'window_size={ws} > series_length={L}.')
        nw=max(0,(L-ws)//wst+1) if ws<=L else 0
        if nw<5: errors.append(f'n_windows={nw} < 5.')
        return {'window_size':ws,'stride':wst,'series_length':L,'n_windows':nw,
                'is_valid':not errors,'errors':errors}


print('validate_lagged_alignment_indices ready.')
print('RollingWindowValidator ready.')


In [ ]:
def expand_score_variants(df):
    rows=[]; has_adj='adjusted_score' in df.columns
    for _,row in df.iterrows():
        base=row.to_dict()
        raw_row={**base,'score_variant':'raw','score_for_ranking':base.get('raw_score',float('nan')),
                 'negative_control_type':'none','negative_control_mean':float('nan'),
                 'negative_control_std':float('nan')}
        rows.append(raw_row)
        if has_adj and pd.notna(base.get('adjusted_score')):
            adj_row={**base,'score_variant':'adjusted',
                     'score_for_ranking':base.get('adjusted_score',float('nan'))}
            rows.append(adj_row)
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=df.columns)


def add_variant_ranks(df):
    df=df.copy()
    gcols=[c for c in ['method_name','layer_spec','pooling_name','score_variant'] if c in df.columns]
    df['rank']=df.groupby(gcols,dropna=False)['score_for_ranking'].rank(
        ascending=False,method='min',na_option='bottom')
    return df


def normalize_classical_df(df):
    df=df.copy()
    if 'score_variant' not in df.columns: df['score_variant']='raw'
    if 'score_for_ranking' not in df.columns and 'score' in df.columns:
        df['score_for_ranking']=df['score']
    if 'raw_score' not in df.columns and 'score' in df.columns:
        df['raw_score']=df['score']
    if 'adjusted_score' not in df.columns: df['adjusted_score']=float('nan')
    if 'layer_spec' not in df.columns: df['layer_spec']='classical'
    if 'pooling_name' not in df.columns: df['pooling_name']='classical'
    return df


def evaluate_all_variants(df, evaluator):
    gcols=[c for c in ['method_name','layer_spec','pooling_name','score_variant',
                        'negative_control_type'] if c in df.columns]
    records=[]
    for keys, grp in df.groupby(gcols, dropna=False):
        if not isinstance(keys,tuple): keys=(keys,)
        kd=dict(zip(gcols,keys))
        cs=grp['causal_setting'].iloc[0] if 'causal_setting' in grp.columns else 'real_traffic_proxy_labels'
        labels=grp['is_causal'].astype(int).values
        scores=grp['score_for_ranking'].fillna(-np.inf).values
        try: m=evaluator._compute_metrics(labels,scores,cs)
        except Exception as e: m={'eval_error':str(e)}
        records.append({**kd,**m,'K':evaluator.K,'n_causal':int(labels.sum()),
                        'n_features':len(labels),'causal_setting':cs})
    return pd.DataFrame(records)


print('Score variant utilities ready: expand_score_variants, add_variant_ranks,')
print('normalize_classical_df, evaluate_all_variants')


## 19. Reporter (Real Traffic)

In [ ]:
import datetime as _dt


class Reporter:
    """
    Saves real-traffic experiment results to disk.
    Output folder: Real_Traffic_Variable_Lag_Target_Relevance_<run_type>_<timestamp>/
    """

    def __init__(self, output_dir='.', run_tag=None):
        self.ts      = run_tag or _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
        self.run_dir = pathlib.Path(output_dir) / f'Real_Traffic_Variable_Lag_Target_Relevance_{self.ts}'
        self._prefix = f'Real_Traffic_Variable_Lag_Analysis_{self.ts}'

    def create_run_folder(self):
        self.run_dir.mkdir(parents=True, exist_ok=True)
        print(f'  Reporter: run folder → {self.run_dir}')
        return self.run_dir

    def save_detailed(self, df):
        p = self.run_dir / f'{self._prefix}_detailed.csv'
        df.to_csv(p, index=False)
        print(f'  Reporter: detailed CSV ({len(df)} rows) → {p.name}')
        return p

    def save_summary(self, df):
        p = self.run_dir / f'{self._prefix}_summary.csv'
        df.to_csv(p, index=False)
        print(f'  Reporter: summary CSV ({len(df)} rows) → {p.name}')
        return p

    def save_runtime(self, d):
        p = self.run_dir / f'{self._prefix}_runtime.csv'
        (pd.DataFrame([d]) if isinstance(d, dict) else d).to_csv(p, index=False)
        print(f'  Reporter: runtime CSV → {p.name}')
        return p

    def save_proxy_diagnostics(self, df):
        if df is None or df.empty: return None
        p = self.run_dir / f'{self._prefix}_proxy_diagnostics.csv'
        df.to_csv(p, index=False)
        print(f'  Reporter: proxy diagnostics CSV ({len(df)} rows) → {p.name}')
        return p

    def save_lag_diagnostics(self, df):
        if df is None or df.empty: return None
        p = self.run_dir / f'{self._prefix}_lag_variability.csv'
        df.to_csv(p, index=False)
        print(f'  Reporter: lag variability CSV ({len(df)} rows) → {p.name}')
        return p

    def save_validation_report(self, checks):
        p = self.run_dir / f'{self._prefix}_validation.csv'
        pd.DataFrame(checks).to_csv(p, index=False)
        n_pass = sum(1 for c in checks if c.get('passed', False))
        print(f'  Reporter: validation CSV ({n_pass}/{len(checks)} passed) → {p.name}')
        return p

    def write_run_metadata(self, metadata):
        import json as _j
        p = self.run_dir / f'{self._prefix}_metadata.json'
        meta = {'notebook_name': CFG.NOTEBOOK_NAME, 'dataset_name': CFG.DATASET_NAME,
                'timestamp': self.ts, **metadata}
        p.write_text(_j.dumps(meta, indent=2, default=str))
        print(f'  Reporter: metadata JSON → {p.name}')
        return p

    @staticmethod
    def build_summary(detailed_df, eval_df=None, group_cols=None):
        if eval_df is None or eval_df.empty: return pd.DataFrame()
        if group_cols is None:
            group_cols = [c for c in [
                'method_name','method_type','T','context_length','noise_level',
                'causal_setting','layer_spec','pooling_name','score_variant',
                'negative_control_type','research_eligible','dcor_status',
                'dcor_implementation','dataset_name',
            ] if c in eval_df.columns]
        agg_metrics = [c for c in [
            'average_precision','precision_at_3','recall_at_3','ndcg_at_3',
            'mean_causal_rank','causal_recovery_at_3','runtime_seconds',
        ] if c in eval_df.columns]
        records=[]
        for keys, grp in eval_df.groupby(group_cols, dropna=False):
            if not isinstance(keys,tuple): keys=(keys,)
            rec=dict(zip(group_cols,keys)); rec['n_instances']=len(grp)
            for col in agg_metrics:
                vals=grp[col].dropna()
                rec[f'{col}_mean']=float(vals.mean()) if len(vals)>0 else float('nan')
                rec[f'{col}_std'] =float(vals.std())  if len(vals)>1 else float('nan')
            records.append(rec)
        return pd.DataFrame(records)


print('Reporter ready (Real Traffic naming).')


## 20. Visualizer (Real Traffic)

Extended with real-traffic specific plots:
- Raw vs centered cosine score range comparison
- Lag variability by feature role
- Proxy Recovery@3 (relabeled from Causal Recovery@3)


In [ ]:
class Visualizer:
    """
    Plotting methods for real-traffic representation metrics analysis.
    Uses 'Proxy Recovery@3' and 'Mean Relevant Rank' in real-traffic plots.
    All methods gracefully skip if matplotlib unavailable or data insufficient.
    """
    _DCOR_LABEL = {'dcor':'dCor (official)','dcor_fallback':'dCor fallback'}

    def __init__(self, output_dir=None):
        self.output_dir = pathlib.Path(output_dir) if output_dir else None
        try: import matplotlib.pyplot; self._plt_ok=True
        except ImportError: self._plt_ok=False
        self._paths=[]

    def _skip(self,name,reason=''):
        print(f'  [Visualizer] Skipping {name}: {reason or "matplotlib unavailable"}')

    def _save(self, fig, filename):
        import matplotlib.pyplot as plt
        if self.output_dir:
            p=self.output_dir/filename; fig.savefig(p,bbox_inches='tight',dpi=150)
            self._paths.append(str(p)); print(f'  [Visualizer] Saved → {p.name}')
        else: plt.show()
        plt.close(fig)

    def _lbl(self,m): return self._DCOR_LABEL.get(m,m)

    def plot_average_precision_by_method(self, summary_df):
        if not self._plt_ok: return self._skip('avg_precision_by_method')
        col='average_precision_mean'
        if col not in summary_df.columns: return self._skip('avg_precision_by_method',f'missing {col}')
        import matplotlib.pyplot as plt
        pivot=summary_df.groupby('method_name')[col].mean().sort_values(ascending=False)
        pivot.index=[self._lbl(m) for m in pivot.index]
        fig,ax=plt.subplots(figsize=(max(6,len(pivot)),4))
        pivot.plot(kind='bar',ax=ax)
        ax.set_ylabel('Mean AP'); ax.set_title('Average Precision by Method')
        ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha='right')
        self._save(fig,'ap_by_method.png')

    def plot_proxy_recovery_by_method(self, summary_df):
        """Proxy Recovery@3 (real-traffic label for causal_recovery_at_3)."""
        if not self._plt_ok: return self._skip('proxy_recovery_by_method')
        col='causal_recovery_at_3_mean'
        if col not in summary_df.columns: return self._skip('proxy_recovery_by_method',f'missing {col}')
        import matplotlib.pyplot as plt
        pivot=summary_df.groupby('method_name')[col].mean().sort_values(ascending=False)
        pivot.index=[self._lbl(m) for m in pivot.index]
        fig,ax=plt.subplots(figsize=(max(6,len(pivot)),4))
        pivot.plot(kind='bar',ax=ax)
        ax.set_ylabel('Proxy Recovery@3'); ax.set_title('Proxy Recovery@3 by Method (real-traffic labels)')
        ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha='right')
        self._save(fig,'proxy_recovery_by_method.png')

    def plot_mean_relevant_rank_by_method(self, summary_df):
        """Mean Relevant Rank (real-traffic label for mean_causal_rank)."""
        if not self._plt_ok: return self._skip('mean_relevant_rank_by_method')
        col='mean_causal_rank_mean'
        if col not in summary_df.columns: return self._skip('mean_relevant_rank_by_method',f'missing {col}')
        import matplotlib.pyplot as plt
        pivot=summary_df.groupby('method_name')[col].mean().sort_values()
        pivot.index=[self._lbl(m) for m in pivot.index]
        fig,ax=plt.subplots(figsize=(max(6,len(pivot)),4))
        pivot.plot(kind='bar',ax=ax)
        ax.set_ylabel('Mean Relevant Rank (lower=better)')
        ax.set_title('Mean Relevant Rank by Method (lower is better)')
        ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha='right')
        self._save(fig,'mean_relevant_rank_by_method.png')

    def plot_ap_layer_heatmap(self, summary_df):
        if not self._plt_ok: return self._skip('ap_layer_heatmap')
        if 'layer_spec' not in summary_df.columns: return self._skip('ap_layer_heatmap','no layer_spec')
        import matplotlib.pyplot as plt
        pivot=summary_df.pivot_table(index='layer_spec',columns='method_name',
                                      values='average_precision_mean',aggfunc='mean')
        if pivot.empty: return self._skip('ap_layer_heatmap','empty pivot')
        pivot.columns=[self._lbl(c) for c in pivot.columns]
        fig,ax=plt.subplots(figsize=(max(6,len(pivot.columns)),max(4,len(pivot))))
        im=ax.imshow(pivot.values,aspect='auto',cmap='YlOrRd')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns,rotation=30,ha='right')
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        plt.colorbar(im,ax=ax,label='Mean AP'); ax.set_title('AP by Layer Spec × Method')
        self._save(fig,'ap_layer_heatmap.png')

    def plot_ap_pooling_heatmap(self, summary_df):
        if not self._plt_ok: return self._skip('ap_pooling_heatmap')
        if 'pooling_name' not in summary_df.columns: return self._skip('ap_pooling_heatmap','no pooling_name')
        import matplotlib.pyplot as plt
        pivot=summary_df.pivot_table(index='pooling_name',columns='method_name',
                                      values='average_precision_mean',aggfunc='mean')
        if pivot.empty: return self._skip('ap_pooling_heatmap','empty pivot')
        pivot.columns=[self._lbl(c) for c in pivot.columns]
        fig,ax=plt.subplots(figsize=(max(6,len(pivot.columns)),max(4,len(pivot))))
        im=ax.imshow(pivot.values,aspect='auto',cmap='YlOrRd')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns,rotation=30,ha='right')
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        plt.colorbar(im,ax=ax,label='Mean AP'); ax.set_title('AP by Pooling × Method')
        self._save(fig,'ap_pooling_heatmap.png')

    def plot_raw_vs_centered_cosine(self, detailed_df):
        """Score range comparison: raw cosine vs centered cosine."""
        if not self._plt_ok: return self._skip('raw_vs_centered_cosine')
        methods=['cosine','centered_cosine']
        sub=detailed_df[detailed_df['method_name'].isin(methods) & detailed_df['raw_score'].notna()]
        if sub.empty: return self._skip('raw_vs_centered_cosine','no cosine or centered_cosine rows')
        import matplotlib.pyplot as plt
        fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=False)
        for ax,mn in zip(axes,methods):
            sdf=sub[sub['method_name']==mn]
            if sdf.empty: ax.set_title(f'{mn} (no data)'); continue
            for role,grp in sdf.groupby('feature_role'):
                ax.hist(grp['raw_score'].dropna(),bins=20,alpha=0.6,label=role)
            ax.set_title(f'{mn}'); ax.set_xlabel('score'); ax.legend(fontsize=7)
        fig.suptitle('Raw vs Centered Cosine Score Distribution by Feature Role')
        fig.tight_layout()
        self._save(fig,'raw_vs_centered_cosine.png')

    def plot_score_dist_by_feature_role(self, detailed_df):
        if not self._plt_ok: return self._skip('score_dist_by_feature_role')
        if 'feature_role' not in detailed_df.columns: return self._skip('score_dist_by_feature_role','no feature_role')
        import matplotlib.pyplot as plt
        tsfm_methods=[m for m in detailed_df['method_name'].unique()
                      if 'cka' in m or 'cosine' in m or 'dcor' in m][:4]
        if not tsfm_methods: return self._skip('score_dist_by_feature_role','no TSFM methods')
        fig,axes=plt.subplots(1,len(tsfm_methods),figsize=(5*len(tsfm_methods),4),squeeze=False)
        for ax,mn in zip(axes[0],tsfm_methods):
            sub=detailed_df[detailed_df['method_name']==mn]
            for role,grp in sub.groupby('feature_role'):
                ax.hist(grp['raw_score'].dropna(),bins=15,alpha=0.6,label=role)
            ax.set_title(self._lbl(mn)); ax.set_xlabel('raw_score'); ax.legend(fontsize=7)
        fig.tight_layout()
        self._save(fig,'score_dist_by_feature_role.png')

    def plot_best_lag_distribution(self, detailed_df):
        if not self._plt_ok: return self._skip('best_lag_distribution')
        if 'best_lag' not in detailed_df.columns: return self._skip('best_lag_distribution','no best_lag')
        sub=detailed_df.dropna(subset=['best_lag'])
        if sub.empty: return self._skip('best_lag_distribution','no lagged metric rows')
        import matplotlib.pyplot as plt
        fig,ax=plt.subplots(figsize=(8,4))
        for role,grp in sub.groupby('feature_role'):
            ax.hist(grp['best_lag'].dropna(),bins=20,alpha=0.6,label=role)
        ax.set_xlabel('best_lag'); ax.set_ylabel('count')
        ax.set_title('Best-Lag Distribution by Feature Role (lagged methods)')
        ax.legend(fontsize=7)
        self._save(fig,'best_lag_distribution.png')

    def plot_runtime_vs_ap(self, summary_df):
        if not self._plt_ok: return self._skip('runtime_vs_ap')
        if ('runtime_seconds_mean' not in summary_df.columns or
            'average_precision_mean' not in summary_df.columns):
            return self._skip('runtime_vs_ap','missing columns')
        import matplotlib.pyplot as plt
        sub=summary_df.dropna(subset=['runtime_seconds_mean','average_precision_mean'])
        if sub.empty: return self._skip('runtime_vs_ap','no valid rows')
        fig,ax=plt.subplots(figsize=(7,5))
        for mn,grp in sub.groupby('method_name'):
            ax.scatter(grp['runtime_seconds_mean'],grp['average_precision_mean'],
                       label=self._lbl(mn),alpha=0.8)
        ax.set_xlabel('Mean Runtime (s)'); ax.set_ylabel('Mean AP')
        ax.set_title('Runtime vs Average Precision'); ax.legend(fontsize=7)
        self._save(fig,'runtime_vs_ap.png')

    def plot_lag_variability_by_role(self, lag_diag_df):
        """Lag variability score distribution by feature role."""
        if not self._plt_ok: return self._skip('lag_variability_by_role')
        if lag_diag_df is None or lag_diag_df.empty:
            return self._skip('lag_variability_by_role','empty lag diagnostics')
        if 'lag_variability_score' not in lag_diag_df.columns:
            return self._skip('lag_variability_by_role','no lag_variability_score')
        import matplotlib.pyplot as plt
        fig,ax=plt.subplots(figsize=(8,4))
        for role,grp in lag_diag_df.groupby('feature_role'):
            ax.hist(grp['lag_variability_score'].dropna(),bins=20,alpha=0.6,label=role)
        ax.set_xlabel('Lag Variability Score (std of windowed best-lag)')
        ax.set_ylabel('count'); ax.set_title('Lag Variability by Feature Role')
        ax.legend(fontsize=7)
        self._save(fig,'lag_variability_by_role.png')

    def saved_paths(self): return list(self._paths)


print('Visualizer ready (10 plots, real-traffic labels).')
print('  Proxy Recovery@3, Mean Relevant Rank, raw vs centered cosine,')
print('  lag variability by role, TSFM score distributions, best-lag distribution.')


## 21. RunConfig (Real Traffic)

In [ ]:
@dataclasses.dataclass
class RunConfig:
    """
    Configuration for a real-traffic experiment run.

    context_lengths      : list of window lengths (T values) in timesteps
    target_sensor_count  : how many target sensors to iterate over
    windows_per_target   : number of rolling windows sampled per (sensor, T)
    run_classical        : compute classical baselines
    run_repr             : compute Chronos representation metrics
    run_lagged           : compute lagged variants
    run_centered_cosine  : compute centered cosine variants
    layer_specs          : list of layer spec strings (e.g. 'layer_6', 'range_3_6_mean')
    pooling_names        : list of pooling names (e.g. 'mean', 'max')
    use_embedding_cache  : cache Chronos embeddings in memory
    checkpoint_interval  : save checkpoint every N instances (0 = disabled)
    run_dir              : override output directory (None = auto)
    run_label            : short label for display / directory name
    dcor_enabled         : run dCor metric (requires dcor package)
    causal_setting       : passed through to Evaluator for compatibility
    """
    context_lengths:       list = dataclasses.field(default_factory=lambda: [168, 336])
    target_sensor_count:   int  = 10
    windows_per_target:    int  = 3
    run_classical:         bool = True
    run_repr:              bool = True
    run_lagged:            bool = True
    run_centered_cosine:   bool = True
    layer_specs:           list = dataclasses.field(default_factory=lambda: [
        'layer_1','layer_3','layer_6','layer_9','layer_11',
        'range_3_6_mean','range_6_9_mean','range_3_10_mean',
    ])
    pooling_names:         list = dataclasses.field(default_factory=lambda: [
        'mean','max','std','norm_weighted',
    ])
    use_embedding_cache:   bool  = True
    checkpoint_interval:   int   = 25
    run_dir:               object= None   # pathlib.Path or None
    run_label:             str   = 'real_traffic'
    dcor_enabled:          bool  = False
    causal_setting:        str   = 'variable_lag'

    # ── derived ─────────────────────────────────────────────────────────
    def total_instances(self, loader: 'MetrLaLoader') -> int:
        n_sensors = min(self.target_sensor_count, len(loader.sensor_ids))
        return n_sensors * len(self.context_lengths) * self.windows_per_target

    def estimate_runtime_seconds(self, loader: 'MetrLaLoader',
                                 secs_per_instance: float = 1.35) -> float:
        layer_factor  = len(self.layer_specs) / 5
        pool_factor   = len(self.pooling_names) / 4
        metric_factor = 1.0 + 0.3 * self.run_lagged + 0.2 * self.run_centered_cosine
        return (self.total_instances(loader)
                * layer_factor * pool_factor * metric_factor
                * secs_per_instance)

    def _parse_layer_ranges(self) -> list:
        """Return list of (lo, hi) tuples from range_* layer specs."""
        ranges = []
        for s in self.layer_specs:
            if s.startswith('range_'):
                parts = s.split('_')          # ['range','3','6','mean']
                ranges.append((int(parts[1]), int(parts[2])))
        return ranges

    def _single_layers(self) -> list:
        return [int(s.split('_')[1]) for s in self.layer_specs
                if s.startswith('layer_')]


## 22. RealTrafficExperimentRunner

In [ ]:
class RealTrafficExperimentRunner:
    """
    Outer loop for real-traffic experiments.

    Iterates over:
        target_sensor  (up to cfg.target_sensor_count)
        context_length (cfg.context_lengths)
        window_idx     (0 .. cfg.windows_per_target - 1)

    For each instance:
        1. Build InstanceData via TrafficVariableLagDatasetBuilder
        2. Run classical baselines (optional)
        3. Run representation metrics via RepresentationScoreRunner (optional)
        4. Collect per-instance rows, checkpoint periodically

    Checkpoint key: "{context_length}|{target_idx}|{window_idx}"
    """

    def __init__(self, cfg: RunConfig, loader: MetrLaLoader,
                 selector: SensorSelector, builder: TrafficVariableLagDatasetBuilder,
                 score_runner: RepresentationScoreRunner):
        self.cfg          = cfg
        self.loader       = loader
        self.selector     = selector
        self.builder      = builder
        self.score_runner = score_runner

    # ── helpers ─────────────────────────────────────────────────────────
    def _ckpt_path(self, run_dir: pathlib.Path) -> pathlib.Path:
        return run_dir / '_checkpoint_detailed.csv'

    def _load_checkpoint(self, run_dir: pathlib.Path):
        p = self._ckpt_path(run_dir)
        if p.exists():
            df = pd.read_csv(p)
            done = set(df['_ckpt_key'].astype(str).tolist())
            print(f'[checkpoint] Resuming — {len(done)} instances already done')
            return df, done
        return pd.DataFrame(), set()

    def _save_checkpoint(self, rows, run_dir: pathlib.Path):
        df = pd.DataFrame(rows)
        df.to_csv(self._ckpt_path(run_dir), index=False)

    # ── main run ────────────────────────────────────────────────────────
    def run(self, run_dir: pathlib.Path = None) -> pd.DataFrame:
        cfg = self.cfg
        if run_dir is None:
            ts  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            run_dir = pathlib.Path(TRAFFIC_CFG.BASE_DIR) / f'{cfg.run_label}_{ts}'
        run_dir.mkdir(parents=True, exist_ok=True)
        print(f'Run dir: {run_dir}')

        prior_df, done_keys = self._load_checkpoint(run_dir)
        rows = prior_df.to_dict('records') if len(prior_df) else []

        sensor_ids = self.loader.sensor_ids[:cfg.target_sensor_count]
        t0         = time.time()
        n_total    = cfg.total_instances(self.loader)
        n_done_at_start = len(done_keys)
        n_new      = 0
        warnings_seen = set()

        for tgt_idx, target_id in enumerate(sensor_ids):
            for ctx_len in cfg.context_lengths:
                for win_idx in range(cfg.windows_per_target):
                    ckpt_key = f'{ctx_len}|{tgt_idx}|{win_idx}'
                    if ckpt_key in done_keys:
                        continue

                    # ── build instance ───────────────────────────────
                    try:
                        group = self.selector.select_for_target(target_id)
                        inst  = self.builder.build_instance(
                            target_sensor_id = target_id,
                            sensor_group     = group,
                            context_length   = ctx_len,
                            window_index     = win_idx,
                        )
                    except Exception as e:
                        wkey = f'build_{target_id}_{ctx_len}_{win_idx}'
                        if wkey not in warnings_seen:
                            print(f'[WARN] build_instance failed ({target_id},{ctx_len},{win_idx}): {e}')
                            warnings_seen.add(wkey)
                        continue

                    base_row = {
                        '_ckpt_key':    ckpt_key,
                        'context_length': ctx_len,
                        'target_id':    target_id,
                        'target_idx':   tgt_idx,
                        'window_idx':   win_idx,
                        'window_start': inst.window_start_time,
                        'causal_setting': cfg.causal_setting,
                    }

                    # ── classical baselines ──────────────────────────
                    if cfg.run_classical:
                        try:
                            cl = ClassicalBaselines(inst)
                            cl_rows = cl.compute_all()
                            for r in cl_rows:
                                rows.append({**base_row, **r, 'method_type': 'classical'})
                        except Exception as e:
                            print(f'[WARN] classical failed ({ckpt_key}): {e}')

                    # ── representation metrics ───────────────────────
                    if cfg.run_repr:
                        try:
                            repr_rows = self.score_runner.compute_all(
                                inst,
                                layer_specs        = cfg.layer_specs,
                                pooling_names      = cfg.pooling_names,
                                run_lagged         = cfg.run_lagged,
                                run_centered_cosine= cfg.run_centered_cosine,
                                dcor_enabled       = cfg.dcor_enabled,
                            )
                            for r in repr_rows:
                                rows.append({**base_row, **r, 'method_type': 'repr'})
                        except Exception as e:
                            print(f'[WARN] repr failed ({ckpt_key}): {e}')

                    done_keys.add(ckpt_key)
                    n_new += 1

                    # ── progress + checkpoint ────────────────────────
                    if cfg.checkpoint_interval > 0 and n_new % cfg.checkpoint_interval == 0:
                        self._save_checkpoint(rows, run_dir)
                        elapsed  = time.time() - t0
                        n_done   = n_done_at_start + n_new
                        rate     = n_new / elapsed if elapsed > 0 else 0
                        rem      = (n_total - n_done) / rate if rate > 0 else 0
                        mem_str  = ''
                        if torch.cuda.is_available():
                            mem_str = (f' | GPU {torch.cuda.memory_allocated()/1e9:.1f}/'
                                       f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
                        print(f'[{n_done}/{n_total}] ctx={ctx_len} tgt={target_id} win={win_idx} '
                              f'| {elapsed/60:.1f}min elapsed | ~{rem/60:.1f}min rem{mem_str}')

        # ── final save ───────────────────────────────────────────────────
        df = pd.DataFrame(rows)
        out_csv = run_dir / 'detailed_scores.csv'
        df.to_csv(out_csv, index=False)
        ckpt = self._ckpt_path(run_dir)
        if ckpt.exists():
            ckpt.unlink()
        elapsed = time.time() - t0
        print(f'Done. {len(df)} rows, {elapsed/60:.1f} min. Saved → {out_csv}')
        return df, run_dir


## 23. Experiment Configs

In [ ]:
# ── Pilot: fast smoke of the full pipeline ──────────────────────────────
REAL_TRAFFIC_PILOT_CONFIG = RunConfig(
    context_lengths      = [168, 336],      # 14h and 28h of 5-min data
    target_sensor_count  = 5,
    windows_per_target   = 2,
    run_classical        = True,
    run_repr             = True,
    run_lagged           = True,
    run_centered_cosine  = True,
    layer_specs          = ['layer_1','layer_3','layer_6','layer_9','layer_11',
                             'range_3_6_mean','range_6_9_mean','range_3_10_mean'],
    pooling_names        = ['mean','max','std','norm_weighted'],
    use_embedding_cache  = True,
    checkpoint_interval  = 10,
    run_label            = 'pilot',
    dcor_enabled         = (_DCOR_AVAILABLE),
    causal_setting       = 'variable_lag',
)

# ── Full Methods: 50 sensors, broader window coverage ───────────────────
REAL_TRAFFIC_FULL_METHODS_CONFIG = RunConfig(
    context_lengths      = [96, 168, 336, 672],   # 8h, 14h, 28h, 56h
    target_sensor_count  = 50,
    windows_per_target   = 3,
    run_classical        = True,
    run_repr             = True,
    run_lagged           = True,
    run_centered_cosine  = True,
    layer_specs          = ['layer_1','layer_3','layer_6','layer_9','layer_11',
                             'range_3_6_mean','range_6_9_mean','range_3_10_mean'],
    pooling_names        = ['mean','max','std','norm_weighted'],
    use_embedding_cache  = True,
    checkpoint_interval  = 25,
    run_label            = 'full_methods',
    dcor_enabled         = (_DCOR_AVAILABLE),
    causal_setting       = 'variable_lag',
)

print('REAL_TRAFFIC_PILOT_CONFIG:')
print(f'  sensors={REAL_TRAFFIC_PILOT_CONFIG.target_sensor_count}',
      f'contexts={REAL_TRAFFIC_PILOT_CONFIG.context_lengths}',
      f'windows/target={REAL_TRAFFIC_PILOT_CONFIG.windows_per_target}')
print()
print('REAL_TRAFFIC_FULL_METHODS_CONFIG:')
print(f'  sensors={REAL_TRAFFIC_FULL_METHODS_CONFIG.target_sensor_count}',
      f'contexts={REAL_TRAFFIC_FULL_METHODS_CONFIG.context_lengths}',
      f'windows/target={REAL_TRAFFIC_FULL_METHODS_CONFIG.windows_per_target}')


## 24. Pilot Run

In [ ]:
# ── Estimate pilot runtime ───────────────────────────────────────────────
pilot_est_sec = REAL_TRAFFIC_PILOT_CONFIG.estimate_runtime_seconds(_loader)
print(f'Pilot estimated runtime: {pilot_est_sec/60:.1f} min')

# ── Build runner ─────────────────────────────────────────────────────────
_pilot_cache   = EmbeddingCache(disk_dir=None) if REAL_TRAFFIC_PILOT_CONFIG.use_embedding_cache else None
_pilot_extractor = ChronosEmbeddingExtractor(
    pipe             = _pipe,
    single_layers    = REAL_TRAFFIC_PILOT_CONFIG._single_layers(),
    layer_ranges     = REAL_TRAFFIC_PILOT_CONFIG._parse_layer_ranges(),
    pooling_names    = REAL_TRAFFIC_PILOT_CONFIG.pooling_names,
    embedding_cache  = _pilot_cache,
)
_pilot_score_runner = RepresentationScoreRunner(
    extractor        = _pilot_extractor,
    embedding_cache  = _pilot_cache,
)
_pilot_runner = RealTrafficExperimentRunner(
    cfg          = REAL_TRAFFIC_PILOT_CONFIG,
    loader       = _loader,
    selector     = _selector,
    builder      = _builder,
    score_runner = _pilot_score_runner,
)

# ── Execute ──────────────────────────────────────────────────────────────
print('Starting pilot run...')
_pilot_df, _pilot_run_dir = _pilot_runner.run()
print(f'Pilot complete. {len(_pilot_df)} rows.')
print(f'Output: {_pilot_run_dir}')


## 25. Pilot Evaluation

In [ ]:
if len(_pilot_df) > 0:
    _pilot_eval = Evaluator(_pilot_df)
    _pilot_summary = _pilot_eval.summary_by_method()
    print('Pilot — Proxy Recovery@3 and Mean Relevant Rank by method:')
    print(_pilot_summary.to_string(index=False))
else:
    print('[WARN] Pilot produced no rows — check logs above.')


## 26. Full Methods Run

In [ ]:
# ── Runtime estimate and confirmation gate ───────────────────────────────
_full_est_sec = REAL_TRAFFIC_FULL_METHODS_CONFIG.estimate_runtime_seconds(_loader)
_full_est_hr  = _full_est_sec / 3600
print(f'Full methods estimated runtime: {_full_est_hr:.1f} h')

_FULL_RUN_THRESHOLD_HOURS = 6.0
if _full_est_hr > _FULL_RUN_THRESHOLD_HOURS:
    print(f'[GATE] Estimated runtime {_full_est_hr:.1f}h exceeds {_FULL_RUN_THRESHOLD_HOURS}h threshold.')
    print('Set _FULL_SUITE_APPROVED = True in the next cell to proceed.')
    _FULL_SUITE_APPROVED = False
else:
    _FULL_SUITE_APPROVED = True
    print(f'[GATE] Estimated runtime {_full_est_hr:.1f}h is within threshold. Auto-approved.')


In [ ]:
# ── Set to True after reviewing the estimate above ───────────────────────
# _FULL_SUITE_APPROVED = True   # uncomment to approve


In [ ]:
if not _FULL_SUITE_APPROVED:
    print('[SKIP] Full run not approved. Set _FULL_SUITE_APPROVED = True to run.')
else:
    _full_cache   = EmbeddingCache(disk_dir=None) if REAL_TRAFFIC_FULL_METHODS_CONFIG.use_embedding_cache else None
    _full_extractor = ChronosEmbeddingExtractor(
        pipe             = _pipe,
        single_layers    = REAL_TRAFFIC_FULL_METHODS_CONFIG._single_layers(),
        layer_ranges     = REAL_TRAFFIC_FULL_METHODS_CONFIG._parse_layer_ranges(),
        pooling_names    = REAL_TRAFFIC_FULL_METHODS_CONFIG.pooling_names,
        embedding_cache  = _full_cache,
    )
    _full_score_runner = RepresentationScoreRunner(
        extractor        = _full_extractor,
        embedding_cache  = _full_cache,
    )
    _full_runner = RealTrafficExperimentRunner(
        cfg          = REAL_TRAFFIC_FULL_METHODS_CONFIG,
        loader       = _loader,
        selector     = _selector,
        builder      = _builder,
        score_runner = _full_score_runner,
    )

    print('Starting full methods run...')
    _full_df, _full_run_dir = _full_runner.run()
    print(f'Full run complete. {len(_full_df)} rows.')
    print(f'Output: {_full_run_dir}')


## 27. Full Run Evaluation + Plots

In [ ]:
if '_full_df' in dir() and len(_full_df) > 0:
    _reporter   = Reporter(run_label='full_methods')
    _viz        = Visualizer()

    _full_eval  = Evaluator(_full_df)
    _full_sum   = _full_eval.summary_by_method()
    print('Full Run — Proxy Recovery@3 and Mean Relevant Rank by method:')
    print(_full_sum.to_string(index=False))

    _reporter.save_summary(_full_sum, tag='full_methods_summary')

    try:
        _viz.plot_proxy_recovery_by_method(_full_df, title='Full Methods — Proxy Recovery@3')
        _viz.plot_mean_relevant_rank_by_method(_full_df, title='Full Methods — Mean Relevant Rank')
        _viz.plot_raw_vs_centered_cosine(_full_df)
        _viz.plot_lag_variability_by_role(_full_df)
    except Exception as _e:
        print(f'[WARN] Plotting failed: {_e}')
else:
    print('[SKIP] Full run did not produce results or was not approved.')


## 28. Ablation Stubs (Future)

The cells below are reserved for ablation runs. Uncomment and configure as needed.

**Layer ablation** — vary single layer index, fixed pooling.
**Pooling ablation** — vary pooling method, fixed layer.
**Context-length ablation** — vary context lengths only.


In [ ]:
# ── Layer ablation config (uncomment to use) ───────────────────────────
# LAYER_ABLATION_CONFIG = RunConfig(
#     context_lengths     = [168],
#     target_sensor_count = 20,
#     windows_per_target  = 3,
#     run_classical       = False,
#     run_repr            = True,
#     layer_specs         = ['layer_1','layer_3','layer_6','layer_9','layer_11'],
#     pooling_names       = ['mean'],
#     run_label           = 'layer_ablation',
# )

# ── Pooling ablation config (uncomment to use) ──────────────────────────
# POOLING_ABLATION_CONFIG = RunConfig(
#     context_lengths     = [168],
#     target_sensor_count = 20,
#     windows_per_target  = 3,
#     run_classical       = False,
#     run_repr            = True,
#     layer_specs         = ['layer_6'],
#     pooling_names       = ['mean','max','std','norm_weighted'],
#     run_label           = 'pooling_ablation',
# )
print('Ablation stubs ready (uncomment to activate).')


## 29. Final Summary

In [ ]:
print('='*70)
print('Real_Traffic_Variable_Lag_Target_Relevance — Session Summary')
print('='*70)
print()
print(f'METR-LA sensors available : {len(_loader.sensor_ids)}')
print(f'Timesteps                 : {len(_loader.timestamps)}')
print(f'Time range                : {_loader.timestamps[0]} → {_loader.timestamps[-1]}')
print()
if '_pilot_df' in dir() and len(_pilot_df) > 0:
    print(f'Pilot rows                : {len(_pilot_df)}')
    print(f'Pilot run dir             : {_pilot_run_dir}')
if '_full_df' in dir() and len(_full_df) > 0:
    print(f'Full run rows             : {len(_full_df)}')
    print(f'Full run dir              : {_full_run_dir}')
print()
print('dCor status               :', _DCOR_STATUS_LABEL)
print('Proxy label note          :', SensorSelector.METADATA_NOTE)
print('='*70)
